**<H1>Contents->QPU, BQM, CQM, DQM, NONLINEAR**

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
dff=dfs_smotes


In [ ]:
dff.info()

<h1>QPU

In [ ]:
# ============================================================
# D-Wave QPU BQM-QSVM — Full Dataset with Per-Sample Timing
# Dataset must already be loaded as: dff
# Target column: LUNG_CANCER
# FIXED: All return signatures match, largest_clique_size removed,
#        DWaveCliqueSampler does NOT receive auto_scale
# ============================================================

!pip install -q --upgrade-strategy only-if-needed dwave-ocean-sdk matplotlib seaborn networkx

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import time
import os
from getpass import getpass

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score,
    roc_curve, precision_recall_curve
)

import dimod
from dimod import BinaryQuadraticModel
from dwave.system import DWaveSampler, EmbeddingComposite, DWaveCliqueSampler
from dwave.cloud import Client

try:
    from dwave.embedding.chain_strength import uniform_torque_compensation
except Exception:
    uniform_torque_compensation = None


# ============================================================
# D-WAVE TOKEN
# ============================================================

DWAVE_TOKEN = getpass("Paste your D-Wave API token: ")


# ============================================================
# QPU SELECTION
# ============================================================

def list_and_select_qpu(token):
    print("\n[QPU SELECTION] Fetching available QPU solvers from D-Wave Leap...")
    _t_start = time.perf_counter()
    try:
        with Client.from_config(token=token) as client:
            solvers = client.get_solvers()
    except Exception as e:
        print(f"[QPU SELECTION] Could not connect: {e}")
        print("[QPU SELECTION] Falling back to: Advantage2_system1")
        return "Advantage2_system1"
    print(f"[TIMING] Solver list fetch: {time.perf_counter() - _t_start:.4f} s")

    qpu_solvers = []
    for s in solvers:
        try:
            props = s.properties
            topo  = props.get("topology", {}).get("type", "")
            if topo.lower() in ("pegasus", "zephyr"):
                clean_name = s.id.split(";")[0].strip()
                qpu_solvers.append({
                    "name":     clean_name,
                    "topology": topo,
                    "qubits":   len(props.get("qubits", [])),
                    "couplers": len(props.get("couplers", [])),
                    "avg_load": props.get("average_load", None),
                    "status":   props.get("status_message", "")
                })
        except Exception:
            continue

    if not qpu_solvers:
        print("[QPU SELECTION] No QPU solvers found. Falling back to: Advantage2_system1")
        return "Advantage2_system1"

    print(f"\n  {'#':<4} {'Name':<35} {'Topology':<10} {'Qubits':<8} {'Couplers':<10} {'Avg Load':<12} Status")
    print("  " + "-" * 90)
    for idx, s in enumerate(qpu_solvers):
        load_str = f"{s['avg_load']:.3f}" if s["avg_load"] is not None else "N/A"
        print(f"  {idx:<4} {s['name']:<35} {s['topology']:<10} {s['qubits']:<8} "
              f"{s['couplers']:<10} {load_str:<12} {s['status']}")

    while True:
        try:
            choice = input(f"\nEnter solver number (0-{len(qpu_solvers)-1}), or Enter for 0: ").strip()
            choice = 0 if choice == "" else int(choice)
            if 0 <= choice < len(qpu_solvers):
                selected = qpu_solvers[choice]["name"]
                print(f"[QPU SELECTION] Selected: {selected}")
                return selected
            else:
                print(f"  Enter 0-{len(qpu_solvers)-1}.")
        except ValueError:
            print("  Invalid input.")


QPU_SOLVER_NAME = list_and_select_qpu(DWAVE_TOKEN)
QPU_SOLVER_NAME = QPU_SOLVER_NAME.split(";")[0].strip()
print(f"\n[QPU SELECTION] Using QPU: {QPU_SOLVER_NAME}")


# ============================================================
# USER CONTROLS
# ============================================================

TEST_SIZE        = 0.8
RANDOM_SEED_BASE = 42
TARGET           = "LUNG_CANCER"
AUTO_SAVE_PATH   = "live_qpu_qsvm_per_sample_results.csv"
n_splits         = 1

N_BITS           = 3
DEFAULT_C        = 0.3
KERNEL_TYPE      = "linear"
RBF_GAMMA        = 1.0
BALANCE_PENALTY  = 5.0

NUM_READS        = 3000
ANNEALING_TIME   = 20
AUTO_SCALE       = True
USE_CLIQUE_SAMPLER_FIRST = True

QPU_TRAIN_SUBSET = 108
VISUALIZE        = True


# ============================================================
# LOAD DATA
# ============================================================

_t_notebook_start = time.perf_counter()
_t_data_load_start = time.perf_counter()

if "dff" not in globals():
    raise ValueError("dff not found.")
if TARGET not in dff.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

X_df          = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()
y_raw         = dff[TARGET].values
unique_targets = sorted(pd.Series(y_raw).dropna().unique())

if set(unique_targets) == {0, 1}:
    y = dff[TARGET].astype(int).values
else:
    if len(unique_targets) != 2:
        raise ValueError(f"Target must be binary. Found: {unique_targets}")
    target_map = {unique_targets[0]: 0, unique_targets[1]: 1}
    y = pd.Series(y_raw).map(target_map).astype(int).values
    print("\nTarget mapping:", target_map)

X_raw = X_df.values.astype(float)
_t_data_load_end = time.perf_counter()
print(f"\n[TIMING] Data load: {_t_data_load_end - _t_data_load_start:.4f} s")
print(dff.info())


# ============================================================
# SELECTED FEATURES
# ============================================================

_t_feat_start = time.perf_counter()
selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]
max_index = max(selected_features_indices)
if max_index >= len(feature_names):
    raise ValueError(f"Feature index {max_index} out of range.")

selected_feature_names = [feature_names[i] for i in selected_features_indices]
X_raw = X_raw[:, selected_features_indices]
_t_feat_end = time.perf_counter()
print(f"\n[TIMING] Feature selection: {_t_feat_end - _t_feat_start:.4f} s")
print("Selected features:", selected_feature_names)
print("Total samples:", X_raw.shape[0], "| Features:", X_raw.shape[1])


# ============================================================
# BASIC FUNCTIONS
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = np.nan
    return {
        "Accuracy":    accuracy_score(y_true, y_pred),
        "Precision":   precision_score(y_true, y_pred, zero_division=0),
        "Recall":      recall_score(y_true, y_pred, zero_division=0),
        "F1":          f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":     roc_auc,
        "PR-AUC":      pr_auc,
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa":       cohen_kappa_score(y_true, y_pred)
    }


def compute_kernel_matrix(XA, XB=None, kernel="linear", rbf_gamma=1.0):
    if XB is None:
        XB = XA
    if kernel == "linear":
        return XA @ XB.T
    elif kernel == "rbf":
        XA_sq = np.sum(XA ** 2, axis=1, keepdims=True)
        XB_sq = np.sum(XB ** 2, axis=1, keepdims=True).T
        dists  = XA_sq + XB_sq - 2 * (XA @ XB.T)
        return np.exp(-rbf_gamma * dists)
    raise ValueError("kernel must be 'linear' or 'rbf'.")


def make_qpu_training_subset(X_train, y_train, subset_size, random_state=42):
    _t_start = time.perf_counter()

    if subset_size is None or subset_size >= len(y_train):
        bqm_vars = len(y_train) * N_BITS
        print(f"  [SUBSET] Using full training set: {len(y_train)} samples "
              f"-> BQM vars = {bqm_vars}")
        if bqm_vars > 150:
            print(f"  [WARNING] BQM vars={bqm_vars} > 150. "
                  f"EmbeddingComposite will likely FAIL. "
                  f"Set QPU_TRAIN_SUBSET <= 50 (N_BITS=3) to fix.")
        print(f"  [TIMING] Subset selection skipped: "
              f"{time.perf_counter() - _t_start:.4f} s")
        return X_train, y_train

    X_sub, _, y_sub, _ = train_test_split(
        X_train, y_train,
        train_size=subset_size,
        stratify=y_train,
        random_state=random_state
    )

    bqm_vars = len(y_sub) * N_BITS
    print(f"  [SUBSET] Stratified QPU subset: {len(y_train)} -> {len(y_sub)} samples "
          f"| BQM vars = {bqm_vars} "
          f"({'OK fits CliqueSampler' if bqm_vars <= 106 else 'OK fits EmbeddingComposite' if bqm_vars <= 150 else 'TOO LARGE'})")
    print(f"  [TIMING] Subset selection: {time.perf_counter() - _t_start:.4f} s")
    return X_sub, y_sub


# ============================================================
# PER-SAMPLE TIMING HELPER
# ============================================================

def per_sample_timing_table(phase_times, n_train, n_test, n_support):
    rows = []
    for phase, info in phase_times.items():
        total_sec   = info["total_sec"]
        denom_label = info["denom_label"]
        n           = info["n"]
        per_sample  = total_sec / n if n > 0 else np.nan
        rows.append({
            "Phase":                  phase,
            "Total_Time_sec":         round(total_sec, 6),
            f"Per_{denom_label}_sec": round(per_sample, 9),
            f"Per_{denom_label}_ms":  round(per_sample * 1e3, 6),
            f"Per_{denom_label}_us":  round(per_sample * 1e6, 3),
            "N_Samples":              n,
            "Denominator":            denom_label
        })
    return pd.DataFrame(rows)


# ============================================================
# BUILD BQM DUAL QSVM WITH DETAILED TIMING
# ============================================================

def build_dual_qsvm_bqm(
    X,
    y,
    n_bits=3,
    C=0.3,
    balance_penalty=5.0,
    kernel="linear",
    rbf_gamma=1.0
):
    _t_total_start = time.perf_counter()

    _t_setup_start = time.perf_counter()
    y_pm        = (2 * y - 1).astype(float)
    n           = X.shape[0]
    q_max       = (2 ** n_bits) - 1
    scale       = C / q_max
    bit_weights = [2 ** k for k in range(n_bits)]
    _t_setup_end = time.perf_counter()
    print(f"  [TIMING] BQM setup/constants: {_t_setup_end - _t_setup_start:.4f} s")

    _t_kernel_start = time.perf_counter()
    K = compute_kernel_matrix(X, kernel=kernel, rbf_gamma=rbf_gamma)
    _t_kernel_end = time.perf_counter()
    t_kernel = _t_kernel_end - _t_kernel_start
    print(f"  [TIMING] Kernel matrix computation ({kernel}, n={n}): {t_kernel:.4f} s")
    print(f"           per sample: {t_kernel/n*1e6:.3f} us")

    _t_bqm_init_start = time.perf_counter()
    bqm = BinaryQuadraticModel({}, {}, 0.0, dimod.BINARY)
    _t_bqm_init_end = time.perf_counter()
    print(f"  [TIMING] Empty BinaryQuadraticModel initialization: "
          f"{_t_bqm_init_end - _t_bqm_init_start:.4f} s")

    def bit_name(i, k):
        return f"a_{i}_{k}"

    def add_term(u, v, coef):
        if abs(coef) < 1e-12:
            return
        if u == v:
            bqm.add_linear(u, coef)
        else:
            bqm.add_quadratic(u, v, coef)

    _t_vars_start = time.perf_counter()
    for i in range(n):
        for k in range(n_bits):
            bqm.add_variable(bit_name(i, k), 0.0)
    _t_vars_end = time.perf_counter()
    print(f"  [TIMING] BQM variable creation ({n * n_bits} variables): "
          f"{_t_vars_end - _t_vars_start:.4f} s")

    _t_linear_start = time.perf_counter()
    for i in range(n):
        for k in range(n_bits):
            bqm.add_linear(bit_name(i, k), -(bit_weights[k] * scale))
    _t_linear_end = time.perf_counter()
    t_linear = _t_linear_end - _t_linear_start
    print(f"  [TIMING] BQM linear objective construction: {t_linear:.4f} s")
    print(f"           per sample: {t_linear/n*1e6:.3f} us")

    _t_quad_start = time.perf_counter()
    for i in range(n):
        for j in range(n):
            base_coef = 0.5 * y_pm[i] * y_pm[j] * K[i, j]
            for ki in range(n_bits):
                for kj in range(n_bits):
                    u   = bit_name(i, ki)
                    v   = bit_name(j, kj)
                    ci  = bit_weights[ki] * scale
                    cj  = bit_weights[kj] * scale
                    coef = base_coef * ci * cj
                    add_term(u, v, coef)
    _t_quad_end = time.perf_counter()
    t_quad = _t_quad_end - _t_quad_start
    print(f"  [TIMING] BQM SVM quadratic objective construction: {t_quad:.4f} s")
    print(f"           per sample-pair: {t_quad/(n*n)*1e6:.4f} us")

    _t_penalty_start = time.perf_counter()
    for i in range(n):
        for j in range(n):
            base_coef = balance_penalty * y_pm[i] * y_pm[j]
            for ki in range(n_bits):
                for kj in range(n_bits):
                    u   = bit_name(i, ki)
                    v   = bit_name(j, kj)
                    ci  = bit_weights[ki] * scale
                    cj  = bit_weights[kj] * scale
                    coef = base_coef * ci * cj
                    add_term(u, v, coef)
    _t_penalty_end = time.perf_counter()
    t_penalty = _t_penalty_end - _t_penalty_start
    print(f"  [TIMING] BQM balance-penalty construction: {t_penalty:.4f} s")
    print(f"           per sample-pair: {t_penalty/(n*n)*1e6:.4f} us")

    _t_total_end = time.perf_counter()
    t_total = _t_total_end - _t_total_start
    print(f"  [TIMING] Total build_dual_qsvm_bqm(): {t_total:.4f} s")
    print(f"           per training sample: {t_total/n*1e6:.3f} us")
    print(f"  [TIMING] Final BQM variables={len(bqm.variables)}, "
          f"interactions={len(bqm.quadratic)}")

    build_timing = {
        "Kernel_sec":  t_kernel,
        "Linear_sec":  t_linear,
        "Quad_sec":    t_quad,
        "Penalty_sec": t_penalty,
        "Total_sec":   t_total,
    }

    return bqm, K, scale, q_max, bit_weights, build_timing


# ============================================================
# QPU TIMING EXTRACTION
# ============================================================

def extract_qpu_timing_details(sampleset, wall_clock_total=None,
                                n_train=None, num_reads=None):
    timing_summary = {}

    try:
        info = sampleset.info if hasattr(sampleset, "info") else {}
    except Exception:
        info = {}

    timing = {}
    try:
        if isinstance(info, dict):
            timing = info.get("timing", {}) or {}
    except Exception:
        timing = {}

    print("\n  [TIMING] D-Wave reported QPU timing breakdown:")
    if timing:
        for k, v in timing.items():
            try:
                v_float = float(v)
                timing_summary[f"DWave_{k}_us"]  = v_float
                timing_summary[f"DWave_{k}_sec"] = v_float / 1e6
                print(f"    {k}: {v_float:.1f} us  ({v_float/1e6:.6f} s)")
            except (TypeError, ValueError):
                timing_summary[f"DWave_{k}"] = v
                print(f"    {k}: {v}")
    else:
        print("    No sampleset.info['timing'] fields returned.")

    key_fields = [
        "qpu_access_time",
        "qpu_sampling_time",
        "qpu_anneal_time_per_sample",
        "qpu_readout_time_per_sample",
        "qpu_programming_time",
        "qpu_delay_time_per_sample",
        "post_processing_overhead_time",
        "total_post_processing_time",
        "run_time",
        "charge_time"
    ]

    print("\n  [TIMING] Key QPU timing fields (if available):")
    for key in key_fields:
        value = timing.get(key, None)
        if value is not None:
            try:
                print(f"    {key}: {value} us  ({float(value)/1e6:.6f} s)")
            except Exception:
                print(f"    {key}: {value}")

    qpu_access_us  = timing.get("qpu_access_time", None)
    qpu_program_us = timing.get("qpu_programming_time", None)
    qpu_sample_us  = timing.get("qpu_sampling_time", None)
    qpu_anneal_us  = timing.get("qpu_anneal_time_per_sample", None)
    qpu_readout_us = timing.get("qpu_readout_time_per_sample", None)
    qpu_delay_us   = timing.get("qpu_delay_time_per_sample", None)
    post_proc_us   = timing.get("total_post_processing_time", None)
    charge_us      = timing.get("charge_time", None)

    print("\n  [TIMING] ===== QPU Detailed Breakdown =====")

    if qpu_access_us is not None:
        qpu_access_sec = float(qpu_access_us) / 1e6
        timing_summary["QPU_Time_sec"] = qpu_access_sec
        print(f"    QPU Access Time (total hardware): "
              f"{float(qpu_access_us):>12.1f} us = {qpu_access_sec:.6f} s")
        if n_train and n_train > 0:
            per_train = qpu_access_sec / n_train
            timing_summary["QPU_Time_per_train_sample_us"] = per_train * 1e6
            print(f"    QPU Access / n_train ({n_train:>4}):   "
                  f"{per_train*1e6:>12.3f} us/sample")
        if num_reads and num_reads > 0:
            per_read = float(qpu_access_us) / num_reads
            timing_summary["QPU_Time_per_read_us"] = per_read
            print(f"    QPU Access / num_reads ({num_reads:>5}): "
                  f"{per_read:>12.3f} us/read")

    if qpu_program_us is not None:
        print(f"    QPU Programming Time:             "
              f"{float(qpu_program_us):>12.1f} us")
        timing_summary["QPU_Programming_us"] = float(qpu_program_us)

    if qpu_sample_us is not None:
        print(f"    QPU Sampling Time:                "
              f"{float(qpu_sample_us):>12.1f} us")

    if qpu_anneal_us is not None:
        t_ann = float(qpu_anneal_us)
        timing_summary["QPU_Anneal_per_read_us"] = t_ann
        print(f"    QPU Anneal Time per Read:         "
              f"{t_ann:>12.1f} us/read")
        if num_reads:
            print(f"    QPU Total Anneal ({num_reads} reads):  "
                  f"{t_ann*num_reads:>12.1f} us = {t_ann*num_reads/1e6:.6f} s")
        if n_train and n_train > 0:
            per_tr = (t_ann * (num_reads or 1)) / n_train
            timing_summary["QPU_Total_Anneal_per_train_sample_us"] = per_tr
            print(f"    QPU Total Anneal / n_train:       "
                  f"{per_tr:>12.3f} us/sample")

    if qpu_readout_us is not None:
        t_ro = float(qpu_readout_us)
        timing_summary["QPU_Readout_per_read_us"] = t_ro
        print(f"    QPU Readout Time per Read:        "
              f"{t_ro:>12.1f} us/read")

    if qpu_delay_us is not None:
        print(f"    QPU Delay Time per Read:          "
              f"{float(qpu_delay_us):>12.1f} us/read")

    if post_proc_us is not None:
        print(f"    Post-Processing Time:             "
              f"{float(post_proc_us):>12.1f} us")

    if charge_us is not None:
        print(f"    Charge Time (billed):             "
              f"{float(charge_us):>12.1f} us = {float(charge_us)/1e6:.4f} s")

    if wall_clock_total is not None:
        timing_summary["Wall_Clock_Submit_To_Result_sec"] = wall_clock_total
        if qpu_access_us is not None:
            try:
                qpu_access_sec = float(qpu_access_us) / 1e6
                overhead_sec   = wall_clock_total - qpu_access_sec
                timing_summary["QPU_Time_sec"]                             = qpu_access_sec
                timing_summary["Queue_Time_Est_sec"]                       = overhead_sec
                timing_summary["Queue_Network_Overhead_sec"]               = overhead_sec
                timing_summary["Estimated_Queue_Network_Embedding_Overhead_sec"] = overhead_sec
                print(f"\n  [TIMING] QPU_Time_sec = qpu_access_time = {qpu_access_sec:.6f} s")
                print(f"  [TIMING] Queue_Time_Est_sec = wall_clock ({wall_clock_total:.4f} s) "
                      f"- qpu_access_time ({qpu_access_sec:.6f} s) = {overhead_sec:.4f} s")
                print("  [TIMING] Note: Queue_Time_Est_sec includes queue + network + "
                      "embedding/client overhead.")
            except Exception:
                pass
        else:
            print(f"\n    Wall-clock total: {wall_clock_total:>12.4f} s")

    try:
        embedding_context = (
            info.get("embedding_context", {})
            if isinstance(info, dict) else {}
        )
        if embedding_context:
            print("\n  [TIMING] Embedding context returned by sampler:")
            print(embedding_context)
            embedding = embedding_context.get("embedding", None)
            if isinstance(embedding, dict) and len(embedding) > 0:
                chain_lengths = [len(chain) for chain in embedding.values()]
                timing_summary["Embedding_Logical_Variables"]    = len(chain_lengths)
                timing_summary["Embedding_Physical_Qubits"]      = int(np.sum(chain_lengths))
                timing_summary["Embedding_Physical_Qubits_Used"] = int(np.sum(chain_lengths))
                timing_summary["Embedding_Max_Chain_Length"]     = int(np.max(chain_lengths))
                timing_summary["Embedding_Max_Chain"]            = int(np.max(chain_lengths))
                timing_summary["Embedding_Mean_Chain_Length"]    = float(np.mean(chain_lengths))
                timing_summary["Embedding_Mean_Chain"]           = float(np.mean(chain_lengths))
                print("\n  [TIMING] Embedding size summary:")
                print("    Logical variables embedded:", timing_summary["Embedding_Logical_Variables"])
                print("    Physical qubits used:", timing_summary["Embedding_Physical_Qubits"])
                print("    Max chain length:", timing_summary["Embedding_Max_Chain_Length"])
                print("    Mean chain length:", timing_summary["Embedding_Mean_Chain_Length"])
                if n_train and n_train > 0:
                    print(f"    Physical qubits / n_train: "
                          f"{int(np.sum(chain_lengths))/n_train:.2f} qubits/sample")
    except Exception as e:
        print(f"\n  [TIMING] Could not extract embedding context: {e}")

    return timing_summary


# ============================================================
# SOLVE BQM ON QPU
# FIXED: No largest_clique_size check (AttributeError risk)
#        DWaveCliqueSampler does NOT receive auto_scale
#        Solver name sanitized inside function
# ============================================================

def solve_bqm_qpu(
    bqm,
    token,
    solver_name,
    num_reads=5000,
    annealing_time=20,
    auto_scale=True,
    use_clique_first=True,
    label="QPU-QSVM",
    n_train=None
):
    solve_timing = {}
    _t_solve_total_start = time.perf_counter()

    # Sanitize solver name
    solver_name_clean = solver_name.split(";")[0].strip()
    if solver_name_clean != solver_name:
        print(f"  [FIX] Sanitized solver name: '{solver_name}' -> '{solver_name_clean}'")
    solver_spec = {"name": solver_name_clean}

    print("\nConnecting to QPU:", solver_name_clean)

    _t_raw_connect_start = time.perf_counter()
    raw_sampler = DWaveSampler(token=token, solver=solver_spec)
    _t_raw_connect_end = time.perf_counter()
    solve_timing["Raw_DWaveSampler_Connection_sec"] = (
        _t_raw_connect_end - _t_raw_connect_start
    )
    print(f"  [TIMING] DWaveSampler connection/solver lookup: "
          f"{solve_timing['Raw_DWaveSampler_Connection_sec']:.4f} s")
    print("Connected QPU solver:", raw_sampler.solver.name)

    _t_solver_props_start = time.perf_counter()
    try:
        print("QPU topology:",
              raw_sampler.properties.get("topology", {}).get("type", "Unknown"))
    except Exception:
        pass
    try:
        print("Active qubits:", len(raw_sampler.nodelist))
    except Exception:
        pass
    _t_solver_props_end = time.perf_counter()
    solve_timing["Solver_Property_Read_sec"] = (
        _t_solver_props_end - _t_solver_props_start
    )
    print(f"  [TIMING] Solver property/nodelist read: "
          f"{solve_timing['Solver_Property_Read_sec']:.4f} s")

    _t_chain_start = time.perf_counter()
    chain_strength = 2.0
    if uniform_torque_compensation is not None:
        try:
            chain_strength = uniform_torque_compensation(bqm)
        except Exception:
            chain_strength = 2.0
    _t_chain_end = time.perf_counter()
    solve_timing["Chain_Strength_Calculation_sec"] = _t_chain_end - _t_chain_start

    print("Using num_reads:", num_reads)
    print("Using annealing_time:", annealing_time)
    print("Using chain_strength:", chain_strength)
    print("Using auto_scale:", auto_scale)
    print(f"  [TIMING] Chain-strength calculation: "
          f"{solve_timing['Chain_Strength_Calculation_sec']:.4f} s")

    # ------------------------------------------------------------
    # Try DWaveCliqueSampler first for dense BQM.
    # FIXED: No largest_clique_size check.
    # FIXED: Do NOT pass auto_scale to DWaveCliqueSampler.sample().
    # ------------------------------------------------------------
    if use_clique_first:
        try:
            print("\nTrying DWaveCliqueSampler for dense BQM embedding...")
            print(f"  BQM size: {len(bqm.variables)} variables")

            _t_clique_init_start = time.perf_counter()
            clique_sampler = DWaveCliqueSampler(
                token=token,
                solver=solver_spec
            )
            _t_clique_init_end = time.perf_counter()
            solve_timing["CliqueSampler_Init_sec"] = (
                _t_clique_init_end - _t_clique_init_start
            )
            print(f"  [TIMING] DWaveCliqueSampler initialization: "
                  f"{solve_timing['CliqueSampler_Init_sec']:.4f} s")

            _t_sample_submit_start = time.perf_counter()
            try:
                sampleset = clique_sampler.sample(
                    bqm,
                    num_reads=num_reads,
                    annealing_time=annealing_time,
                    chain_strength=chain_strength,
                    label=label
                )
            except TypeError:
                print("DWaveCliqueSampler did not accept chain_strength.")
                print("Retrying without chain_strength...")
                sampleset = clique_sampler.sample(
                    bqm,
                    num_reads=num_reads,
                    annealing_time=annealing_time,
                    label=label
                )
            _t_sample_return = time.perf_counter()

            _t_resolve_start = time.perf_counter()
            try:
                if hasattr(sampleset, "resolve"):
                    sampleset.resolve()
            except Exception:
                pass
            # Force result fetch
            _ = sampleset.first
            _t_resolve_end = time.perf_counter()

            solver_used = "DWaveCliqueSampler"
            solve_timing["Sampler_Sample_Call_Return_sec"] = (
                _t_sample_return - _t_sample_submit_start
            )
            solve_timing["Sampleset_Resolve_sec"] = (
                _t_resolve_end - _t_resolve_start
            )
            solve_timing["Wall_Clock_Submit_To_Result_sec"] = (
                _t_resolve_end - _t_sample_submit_start
            )
            solve_timing["Wall_Clock_sec"] = (
                solve_timing["Wall_Clock_Submit_To_Result_sec"]
            )

            print("QPU sampling completed using DWaveCliqueSampler.")
            print(f"  [TIMING] sampler.sample() call returned: "
                  f"{solve_timing['Sampler_Sample_Call_Return_sec']:.4f} s")
            print(f"  [TIMING] sampleset.resolve()/blocking fetch: "
                  f"{solve_timing['Sampleset_Resolve_sec']:.4f} s")
            print(f"  [TIMING] Wall-clock submit to result resolved: "
                  f"{solve_timing['Wall_Clock_Submit_To_Result_sec']:.4f} s")

            qpu_timing = extract_qpu_timing_details(
                sampleset,
                wall_clock_total=solve_timing["Wall_Clock_Submit_To_Result_sec"],
                n_train=n_train,
                num_reads=num_reads
            )
            solve_timing.update(qpu_timing)

            _t_solve_total_end = time.perf_counter()
            solve_timing["Total_solve_bqm_qpu_sec"] = (
                _t_solve_total_end - _t_solve_total_start
            )
            solve_timing["Total_sec"] = solve_timing["Total_solve_bqm_qpu_sec"]
            print(f"  [TIMING] Total solve_bqm_qpu(): "
                  f"{solve_timing['Total_solve_bqm_qpu_sec']:.4f} s")

            return sampleset, solver_used, chain_strength, solve_timing

        except Exception as e:
            _t_clique_fail_end = time.perf_counter()
            solve_timing["CliqueSampler_Failed_After_sec"] = (
                _t_clique_fail_end - _t_solve_total_start
            )
            print("\nDWaveCliqueSampler failed.")
            print("Reason:", e)
            print(f"  [TIMING] DWaveCliqueSampler failed after: "
                  f"{solve_timing['CliqueSampler_Failed_After_sec']:.4f} s")
            print("Falling back to EmbeddingComposite(DWaveSampler).")

    # ------------------------------------------------------------
    # Fallback: EmbeddingComposite
    # ------------------------------------------------------------
    bqm_size = len(bqm.variables)
    if bqm_size > 150:
        print(f"\n  [WARNING] BQM size={bqm_size} > 150. "
              f"EmbeddingComposite may also fail to embed on Advantage2. "
              f"Reduce QPU_TRAIN_SUBSET if this errors.")

    _t_embedding_init_start = time.perf_counter()
    sampler = EmbeddingComposite(raw_sampler)
    _t_embedding_init_end = time.perf_counter()
    solve_timing["EmbeddingComposite_Init_sec"] = (
        _t_embedding_init_end - _t_embedding_init_start
    )
    print(f"  [TIMING] EmbeddingComposite initialization: "
          f"{solve_timing['EmbeddingComposite_Init_sec']:.4f} s")

    _t_sample_submit_start = time.perf_counter()
    try:
        sampleset = sampler.sample(
            bqm,
            num_reads=num_reads,
            annealing_time=annealing_time,
            chain_strength=chain_strength,
            auto_scale=auto_scale,
            label=label
        )
    except TypeError:
        print("\nEmbeddingComposite did not accept auto_scale.")
        print("Retrying without auto_scale...")
        sampleset = sampler.sample(
            bqm,
            num_reads=num_reads,
            annealing_time=annealing_time,
            chain_strength=chain_strength,
            label=label
        )
    _t_sample_return = time.perf_counter()

    _t_resolve_start = time.perf_counter()
    try:
        if hasattr(sampleset, "resolve"):
            sampleset.resolve()
    except Exception:
        pass
    # Force result fetch
    _ = sampleset.first
    _t_resolve_end = time.perf_counter()

    solver_used = "EmbeddingComposite(DWaveSampler)"
    solve_timing["Sampler_Sample_Call_Return_sec"] = (
        _t_sample_return - _t_sample_submit_start
    )
    solve_timing["Sampleset_Resolve_sec"] = (
        _t_resolve_end - _t_resolve_start
    )
    solve_timing["Wall_Clock_Submit_To_Result_sec"] = (
        _t_resolve_end - _t_sample_submit_start
    )
    solve_timing["Wall_Clock_sec"] = (
        solve_timing["Wall_Clock_Submit_To_Result_sec"]
    )

    print("QPU sampling completed using EmbeddingComposite.")
    print(f"  [TIMING] sampler.sample() call returned: "
          f"{solve_timing['Sampler_Sample_Call_Return_sec']:.4f} s")
    print(f"  [TIMING] sampleset.resolve()/blocking fetch: "
          f"{solve_timing['Sampleset_Resolve_sec']:.4f} s")
    print(f"  [TIMING] Wall-clock submit to result resolved: "
          f"{solve_timing['Wall_Clock_Submit_To_Result_sec']:.4f} s")

    qpu_timing = extract_qpu_timing_details(
        sampleset,
        wall_clock_total=solve_timing["Wall_Clock_Submit_To_Result_sec"],
        n_train=n_train,
        num_reads=num_reads
    )
    solve_timing.update(qpu_timing)

    _t_solve_total_end = time.perf_counter()
    solve_timing["Total_solve_bqm_qpu_sec"] = (
        _t_solve_total_end - _t_solve_total_start
    )
    solve_timing["Total_sec"] = solve_timing["Total_solve_bqm_qpu_sec"]
    print(f"  [TIMING] Total solve_bqm_qpu(): "
          f"{solve_timing['Total_solve_bqm_qpu_sec']:.4f} s")

    return sampleset, solver_used, chain_strength, solve_timing


# ============================================================
# EXTRACT ALPHA, OBJECTIVE, WEIGHT, BIAS
# FIXED: All return signatures match first script caller expectations
# ============================================================

def extract_alpha_from_bqm_sample(sample, n, n_bits, scale, bit_weights):
    q_values = np.zeros(n)
    for i in range(n):
        q_i = 0
        for k in range(n_bits):
            q_i += bit_weights[k] * int(sample.get(f"a_{i}_{k}", 0))
        q_values[i] = q_i
    alpha = q_values * scale
    return q_values, alpha


def true_svm_dual_objective(alpha, y_train, K_train):
    y_pm = (2 * y_train - 1).astype(float)
    quad = 0.0
    for i in range(len(alpha)):
        for j in range(len(alpha)):
            quad += (
                0.5 * alpha[i] * alpha[j]
                * y_pm[i] * y_pm[j] * K_train[i, j]
            )
    return quad - np.sum(alpha)


def select_best_qpu_solution(
    sampleset,
    n_train,
    n_bits,
    scale,
    bit_weights,
    y_train,
    K_train,
    balance_tolerance=1e-9
):
    _t_select_start = time.perf_counter()
    y_pm = (2 * y_train - 1).astype(float)
    candidates = []

    for row in sampleset.data(["sample", "energy", "num_occurrences"]):
        sample = row.sample
        energy = row.energy
        num_occurrences = row.num_occurrences

        q_values, alpha = extract_alpha_from_bqm_sample(
            sample, n=n_train, n_bits=n_bits,
            scale=scale, bit_weights=bit_weights
        )

        balance_value = float(np.sum(alpha * y_pm))
        balance_abs   = abs(balance_value)
        true_obj      = true_svm_dual_objective(alpha, y_train, K_train)
        nonzero_alpha_count = int(np.sum(alpha > 1e-8))

        candidates.append({
            "sample":              sample,
            "energy":              energy,
            "num_occurrences":     num_occurrences,
            "q_values":            q_values,
            "alpha":               alpha,
            "balance_value":       balance_value,
            "balance_abs":         balance_abs,
            "true_svm_objective":  true_obj,
            "nonzero_alpha_count": nonzero_alpha_count
        })

    exact_balance = [
        c for c in candidates
        if c["balance_abs"] <= balance_tolerance
        and c["nonzero_alpha_count"] > 0
    ]

    if len(exact_balance) > 0:
        best = min(exact_balance, key=lambda c: c["true_svm_objective"])
        selection_note = "Exact-balanced QPU solution selected."
    else:
        best = min(candidates, key=lambda c: (c["balance_abs"], c["energy"]))
        selection_note = "No exact-balanced solution found. Selected smallest balance violation."

    t_select = time.perf_counter() - _t_select_start
    print(f"  [TIMING] Candidate extraction + best QPU solution selection "
          f"({len(candidates)} returned samples): {t_select:.4f} s")

    # FIXED: returns 4 values — best, candidates, selection_note, t_select
    return best, candidates, selection_note, t_select


def compute_bias_dual(alpha, y_train, K_train, C=0.3):
    _t_bias_start = time.perf_counter()
    y_pm = (2 * y_train - 1).astype(float)
    support_indices = np.where(alpha > 1e-8)[0]
    margin_indices  = np.where((alpha > 1e-8) & (alpha < C - 1e-8))[0]
    selected = margin_indices if len(margin_indices) > 0 else support_indices

    if len(selected) == 0:
        t_bias = time.perf_counter() - _t_bias_start
        print(f"  [TIMING] Bias computation (no support vectors): {t_bias:.4f} s")
        # FIXED: returns 3 values — b, support_indices, t_bias
        return 0.0, support_indices, t_bias

    b_values = []
    for i in selected:
        decision_without_b = np.sum(alpha * y_pm * K_train[i, :])
        b_i = y_pm[i] - decision_without_b
        b_values.append(b_i)
    b = float(np.mean(b_values))

    t_bias = time.perf_counter() - _t_bias_start
    print(f"  [TIMING] Bias computation ({len(selected)} vectors used): {t_bias:.4f} s")
    print(f"           per support vector: {t_bias/len(selected)*1e6:.3f} us")

    # FIXED: returns 3 values — b, support_indices, t_bias
    return b, support_indices, t_bias


def decision_function_dual(
    X_query, X_train, alpha, y_train, b,
    kernel="linear", rbf_gamma=1.0
):
    y_pm    = (2 * y_train - 1).astype(float)
    K_query = compute_kernel_matrix(
        X_query, X_train, kernel=kernel, rbf_gamma=rbf_gamma
    )
    return K_query @ (alpha * y_pm) + b


def predict_probability_dual(
    X_query, X_train, alpha, y_train, b,
    kernel="linear", rbf_gamma=1.0, label=""
):
    n_query = X_query.shape[0]
    n_train = X_train.shape[0]

    tag = f" [{label}]" if label else ""

    _t_kernel_start = time.perf_counter()
    y_pm    = (2 * y_train - 1).astype(float)
    K_query = compute_kernel_matrix(
        X_query, X_train, kernel=kernel, rbf_gamma=rbf_gamma
    )
    t_kernel_pred = time.perf_counter() - _t_kernel_start

    _t_score_start = time.perf_counter()
    scores = K_query @ (alpha * y_pm) + b
    probs  = sigmoid(scores)
    t_score = time.perf_counter() - _t_score_start

    t_total_pred = t_kernel_pred + t_score

    print(f"  [TIMING] Prediction{tag} total:          "
          f"{t_total_pred:.6f} s  "
          f"(n_query={n_query}, n_train={n_train})")
    print(f"    Kernel eval:                        "
          f"{t_kernel_pred:.6f} s  "
          f"|  per query sample: {t_kernel_pred/n_query*1e6:.3f} us")
    print(f"    Score + sigmoid:                    "
          f"{t_score:.6f} s  "
          f"|  per query sample: {t_score/n_query*1e6:.3f} us")
    print(f"    Combined per query sample:          "
          f"{t_total_pred/n_query*1e6:.3f} us")
    print(f"    Combined per train sample (kernel): "
          f"{t_kernel_pred/n_train*1e6:.3f} us")

    # FIXED: returns 4 values — probs, t_total_pred, t_kernel_pred, t_score
    return probs, t_total_pred, t_kernel_pred, t_score


def extract_linear_weight_from_alpha(alpha, X_train, y_train):
    _t_start = time.perf_counter()
    y_pm = (2 * y_train - 1).astype(float)
    w    = (alpha * y_pm) @ X_train
    t_w  = time.perf_counter() - _t_start
    print(f"  [TIMING] Linear weight vector extraction: {t_w:.6f} s")
    # FIXED: returns 2 values — w, t_w
    return w, t_w


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_per_sample_timing(per_sample_df,
                            title="Per-Sample Timing Breakdown"):
    us_cols = [c for c in per_sample_df.columns if c.endswith("_us")]
    if not us_cols:
        print("No per-sample us columns found.")
        return
    col = us_cols[0]
    df_plot = (
        per_sample_df[["Phase", col]]
        .dropna()
        .pipe(lambda df: df[df[col] > 0])
        .sort_values(col, ascending=False)
    )
    plt.figure(figsize=(11, 5))
    bars = plt.bar(
        df_plot["Phase"], df_plot[col],
        color=sns.color_palette("muted", len(df_plot))
    )
    plt.yscale("log")
    for bar, val in zip(bars, df_plot[col]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.15,
            f"{val:.2f}", ha="center", va="bottom",
            fontsize=8, rotation=45
        )
    plt.title(title)
    plt.xlabel("Phase")
    plt.ylabel("Time per Sample (us, log scale)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_train_vs_test_per_sample(train_us, test_us, n_train, n_test):
    labels = [
        "Train Inference\n(per train sample)",
        "Test Inference\n(per test sample)"
    ]
    values = [train_us, test_us]
    colors = ["steelblue", "darkorange"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(labels, values, color=colors)
    for i, v in enumerate(values):
        axes[0].text(i, v * 1.02, f"{v:.3f} us", ha="center", fontsize=10)
    axes[0].set_title("Prediction Time per Sample (us)")
    axes[0].set_ylabel("Time (us)")
    axes[0].grid(axis="y", alpha=0.3)
    totals = [train_us * n_train / 1e6, test_us * n_test / 1e6]
    axes[1].bar(
        [f"Train (n={n_train})", f"Test (n={n_test})"],
        totals, color=colors
    )
    for i, v in enumerate(totals):
        axes[1].text(i, v * 1.02, f"{v:.4f} s", ha="center", fontsize=10)
    axes[1].set_title("Total Inference Time (s)")
    axes[1].set_ylabel("Time (s)")
    axes[1].grid(axis="y", alpha=0.3)
    plt.suptitle("Training vs Testing Inference Time", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_qpu_per_sample_breakdown(solve_timing, n_train, num_reads):
    components = {}
    qpu_prog = solve_timing.get("QPU_Programming_us", None)
    qpu_ann  = solve_timing.get("QPU_Anneal_per_read_us", None)
    qpu_ro   = solve_timing.get("QPU_Readout_per_read_us", None)
    overhead = solve_timing.get("Queue_Network_Overhead_sec", None)
    if qpu_prog is not None:
        components["Programming"]       = float(qpu_prog) / n_train
    if qpu_ann is not None:
        components["Anneal x num_reads"] = float(qpu_ann) * num_reads / n_train
    if qpu_ro is not None:
        components["Readout x num_reads"] = float(qpu_ro) * num_reads / n_train
    if overhead is not None:
        components["Queue+Network overhead"] = overhead * 1e6 / n_train
    if not components:
        print("Insufficient QPU timing data for per-sample breakdown plot.")
        return
    plt.figure(figsize=(9, 5))
    bars = plt.bar(
        list(components.keys()), list(components.values()),
        color=sns.color_palette("Set2", len(components))
    )
    for bar, val in zip(bars, components.values()):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f"{val:.2f} us", ha="center", va="bottom", fontsize=9
        )
    plt.title(f"QPU Time Components per Training Sample (n_train={n_train})")
    plt.ylabel("Time per Training Sample (us)")
    plt.xlabel("QPU Phase")
    plt.yscale("log")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_qpu_energy_convergence(sampleset):
    energies = np.asarray(sampleset.record.energy)
    best = np.minimum.accumulate(energies)
    plt.figure(figsize=(7, 4))
    plt.plot(best, linewidth=1)
    plt.title("Advantage2 QPU Energy Convergence")
    plt.xlabel("Returned Sample Index")
    plt.ylabel("Best Energy So Far")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_qpu_energy_hist(sampleset):
    energies = np.asarray(sampleset.record.energy)
    plt.figure(figsize=(7, 4))
    plt.hist(energies, bins=30)
    plt.title("Advantage2 QPU Energy Distribution")
    plt.xlabel("Energy")
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_qpu_solution_frequency(sampleset, top_k=10):
    try:
        df_samples = sampleset.to_pandas_dataframe()
    except Exception as e:
        print("Cannot convert sampleset to dataframe:", e)
        return
    var_cols = [
        col for col in df_samples.columns
        if str(col).startswith("a_")
    ]
    if len(var_cols) == 0:
        print("No alpha bit variables found for solution frequency plot.")
        return
    if "num_occurrences" not in df_samples.columns:
        df_samples["num_occurrences"] = 1
    df_samples["_solution_signature"] = (
        df_samples[var_cols]
        .astype(int).astype(str)
        .agg(",".join, axis=1)
    )
    counts = (
        df_samples
        .groupby("_solution_signature")["num_occurrences"]
        .sum()
        .sort_values(ascending=False)
        .head(top_k)
    )
    short_labels = [
        s[:80] + "..." if len(s) > 80 else s
        for s in counts.index
    ]
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(counts)), counts.values)
    plt.xticks(range(len(counts)), short_labels, rotation=45, ha="right")
    plt.title(f"Top {top_k} QPU Solution Frequencies")
    plt.ylabel("Occurrences")
    plt.tight_layout()
    plt.show()


def plot_qubo_heatmap(bqm, max_variables=120):
    all_vars  = list(bqm.variables)
    var_order = all_vars[:max_variables] if len(all_vars) > max_variables else all_vars
    var_index = {v: i for i, v in enumerate(var_order)}
    mat = np.zeros((len(var_order), len(var_order)))
    for v in var_order:
        mat[var_index[v], var_index[v]] = bqm.get_linear(v)
    for (u, v), bias in bqm.quadratic.items():
        if u in var_index and v in var_index:
            i, j = var_index[u], var_index[v]
            mat[i, j] = mat[j, i] = bias
    plt.figure(figsize=(8, 7))
    sns.heatmap(mat, cmap="coolwarm", center=0,
                xticklabels=False, yticklabels=False)
    plt.title("BQM/QUBO Matrix Heatmap")
    plt.xlabel("Binary Variables")
    plt.ylabel("Binary Variables")
    plt.show()


def plot_qubo_graph(bqm, max_edges=100):
    edges = sorted(
        [(u, v, abs(b)) for (u, v), b in bqm.quadratic.items() if abs(b) > 0],
        key=lambda x: x[2], reverse=True
    )[:max_edges]
    if not edges:
        print("No quadratic interactions found in BQM.")
        return
    H = nx.Graph()
    for u, v, weight in edges:
        H.add_edge(u, v, weight=weight)
    plt.figure(figsize=(9, 7))
    nx.draw(
        H, nx.spring_layout(H, seed=42),
        node_color="lightblue", edge_color="gray",
        with_labels=True, node_size=450, font_size=7
    )
    plt.title(f"BQM/QUBO Interaction Graph: Top {max_edges} Strongest Edges")
    plt.show()


def plot_kernel_matrix_heatmap(K, title="Training Kernel Matrix"):
    plt.figure(figsize=(7, 6))
    sns.heatmap(K, cmap="viridis")
    plt.title(title)
    plt.xlabel("Training Sample Index")
    plt.ylabel("Training Sample Index")
    plt.show()


def plot_dual_objective_matrix(K, y_train, scale):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)
    plt.figure(figsize=(7, 6))
    sns.heatmap(Q, cmap="coolwarm", center=0)
    plt.title("Dual QSVM Interaction Matrix")
    plt.xlabel("Training Sample j")
    plt.ylabel("Training Sample i")
    plt.show()


def plot_alpha_values(alpha):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(alpha)), alpha)
    plt.title("Optimized Alpha Values")
    plt.xlabel("Training Sample Index")
    plt.ylabel("Alpha")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_alpha_by_class(alpha, y_train):
    df_alpha = pd.DataFrame({
        "Alpha": alpha,
        "Class": y_train.astype(int)
    })
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_alpha, x="Class", y="Alpha")
    sns.stripplot(data=df_alpha, x="Class", y="Alpha", alpha=0.7)
    plt.title("Alpha Distribution by Class")
    plt.xlabel("Class")
    plt.ylabel("Alpha")
    plt.show()


def plot_support_vectors_pca(X_train, y_train, alpha):
    support_idx = alpha > 1e-8
    X_vis = (
        PCA(n_components=2).fit_transform(X_train)
        if X_train.shape[1] > 2 else X_train
    )
    plt.figure(figsize=(7, 5))
    plt.scatter(X_vis[:, 0], X_vis[:, 1],
                c=y_train, cmap="bwr", s=50, label="Training Samples")
    plt.scatter(X_vis[support_idx, 0], X_vis[support_idx, 1],
                s=160, facecolors="none", edgecolors="black",
                linewidths=1.5, label="Support Vectors")
    plt.title("Support Vectors in PCA Space")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_weight_importance(w, selected_feature_names):
    importance = np.abs(w)
    df_imp = pd.DataFrame({
        "Feature":    selected_feature_names,
        "Weight":     w,
        "Abs_Weight": importance
    }).sort_values("Abs_Weight", ascending=False)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=df_imp, x="Abs_Weight", y="Feature")
    plt.title("Feature Importance from QPU-BQM-QSVM Linear Weight Vector")
    plt.xlabel("|Weight|")
    plt.ylabel("Feature")
    plt.show()
    return df_imp


def plot_confusion_matrix_custom(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred 0", "Pred 1"],
                yticklabels=["Actual 0", "Actual 1"])
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def plot_roc_curve_custom(y_true, y_prob):
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_score   = roc_auc_score(y_true, y_prob)
    except Exception as e:
        print("ROC curve cannot be plotted:", e)
        return
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_pr_curve_custom(y_true, y_prob):
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap_score             = average_precision_score(y_true, y_prob)
    except Exception as e:
        print("PR curve cannot be plotted:", e)
        return
    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC/AP = {ap_score:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_prediction_distribution(y_true, y_prob):
    df_pred = pd.DataFrame({
        "Probability": y_prob,
        "Class":       y_true.astype(int)
    })
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_pred, x="Probability", hue="Class", bins=30, kde=True)
    plt.title("Predicted Probability Distribution")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.show()


def plot_metrics_bar(metrics):
    selected_metrics = {
        "Accuracy":    metrics.get("Accuracy", np.nan),
        "Precision":   metrics.get("Precision", np.nan),
        "Recall":      metrics.get("Recall", np.nan),
        "F1":          metrics.get("F1", np.nan),
        "ROC-AUC":     metrics.get("ROC-AUC", np.nan),
        "PR-AUC":      metrics.get("PR-AUC", np.nan),
        "Sensitivity": metrics.get("Sensitivity", np.nan),
        "Specificity": metrics.get("Specificity", np.nan),
        "Kappa":       metrics.get("Kappa", np.nan)
    }
    names  = list(selected_metrics.keys())
    values = list(selected_metrics.values())
    plt.figure(figsize=(10, 4))
    plt.bar(names, values)
    plt.ylim(0, 1.05)
    plt.title("Model Performance Metrics")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_decision_boundary_pca_dual(
    X_train, y_train, alpha, b,
    kernel="linear", rbf_gamma=1.0
):
    if X_train.shape[1] > 2:
        pca   = PCA(n_components=2)
        X_vis = pca.fit_transform(X_train)
    else:
        pca   = None
        X_vis = X_train

    x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
    y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    grid_2d   = np.c_[xx.ravel(), yy.ravel()]
    grid_full = pca.inverse_transform(grid_2d) if pca is not None else grid_2d

    scores = decision_function_dual(
        grid_full, X_train, alpha, y_train, b,
        kernel=kernel, rbf_gamma=rbf_gamma
    )
    z = sigmoid(scores).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, z, levels=50, alpha=0.8)
    plt.scatter(X_vis[:, 0], X_vis[:, 1],
                c=y_train, cmap="bwr", edgecolors="black", s=45)
    plt.title("Decision Boundary in PCA Space")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(label="Predicted Probability")
    plt.show()


def plot_runtime_summary(summary_df):
    if "T_Total_sec" not in summary_df.columns:
        print("T_Total_sec column not found.")
        return
    plt.figure(figsize=(8, 4))
    sns.barplot(data=summary_df, x="Split", y="T_Total_sec")
    plt.title("Runtime per Split")
    plt.xlabel("Split")
    plt.ylabel("Runtime (s)")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def run_all_qpu_visualizations(
    bqm, sampleset, K_train,
    X_train, y_train,
    X_test, y_test,
    y_prob_test, y_prob_train,
    alpha, w, b,
    metrics_test, metrics_train,
    selected_feature_names,
    scale, kernel, rbf_gamma,
    solve_timing, n_train, num_reads
):
    print("\n========== Advantage2 QPU / Solver Visualizations ==========")
    plot_qpu_energy_convergence(sampleset)
    plot_qpu_energy_hist(sampleset)
    plot_qpu_solution_frequency(sampleset, top_k=10)

    print("\n========== BQM / QUBO Structure Visualizations ==========")
    plot_qubo_heatmap(bqm, max_variables=120)
    plot_qubo_graph(bqm, max_edges=100)

    print("\n========== Kernel / Objective Visualizations ==========")
    plot_kernel_matrix_heatmap(K_train, title="Training Kernel Matrix")
    plot_dual_objective_matrix(K_train, y_train, scale)

    print("\n========== Alpha / Support Vector Visualizations ==========")
    plot_alpha_values(alpha)
    plot_alpha_by_class(alpha, y_train)
    plot_support_vectors_pca(X_train, y_train, alpha)

    print("\n========== Model Performance Visualizations (Test Set) ==========")
    plot_confusion_matrix_custom(y_test, y_prob_test, threshold=0.5)
    plot_roc_curve_custom(y_test, y_prob_test)
    plot_pr_curve_custom(y_test, y_prob_test)
    plot_prediction_distribution(y_test, y_prob_test)
    plot_metrics_bar(metrics_test)

    print("\n========== Feature / Decision Boundary Visualizations ==========")
    feature_importance_df = None
    if kernel == "linear" and w is not None:
        feature_importance_df = plot_weight_importance(w, selected_feature_names)
    else:
        print("Feature weight importance only available for linear kernel.")

    plot_decision_boundary_pca_dual(
        X_train, y_train, alpha, b,
        kernel=kernel, rbf_gamma=rbf_gamma
    )

    print("\n========== Per-Sample Timing Visualizations ==========")
    n_test = X_test.shape[0]
    n_sv   = int(np.sum(alpha > 1e-8))

    phase_times_vis = {}
    t_train_pred = X_train.shape[0]
    t_test_pred  = X_test.shape[0]

    plot_qpu_per_sample_breakdown(solve_timing, n_train, num_reads)

    return feature_importance_df


# ============================================================
# MAIN LOOP — FULL DATASET + PER-SAMPLE TIMING
# ============================================================

summary = []
_t_notebook_main = time.perf_counter()

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

cfg = dict(
    kernel=KERNEL_TYPE,
    C=DEFAULT_C,
    n_bits=N_BITS,
    rbf_gamma=RBF_GAMMA,
    balance_penalty=BALANCE_PENALTY
)

for split in range(1, n_splits + 1):

    print("\n" + "=" * 70)
    print(f"SPLIT {split}/{n_splits}  --  FULL DATASET RUN")
    print("=" * 70)

    _t_split_start = time.perf_counter()

    # ---- Split ----
    _t_ds = time.perf_counter()
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * 42
    )
    t_split = time.perf_counter() - _t_ds
    print(f"[TIMING] train_test_split: {t_split:.4f} s  "
          f"| n_train_full={X_train_full.shape[0]}, n_test={X_test.shape[0]}")

    # ---- Preprocessing ----
    _t_prep = time.perf_counter()
    imp = SimpleImputer(strategy="median")
    sc  = StandardScaler()
    X_train_full = sc.fit_transform(imp.fit_transform(X_train_full))
    X_test       = sc.transform(imp.transform(X_test))
    t_prep       = time.perf_counter() - _t_prep
    n_train_full = X_train_full.shape[0]
    n_test       = X_test.shape[0]
    print(f"[TIMING] Preprocessing: {t_prep:.4f} s  "
          f"| per train sample: {t_prep/n_train_full*1e6:.3f} us  "
          f"| per test sample: {t_prep/n_test*1e6:.3f} us")

    # ---- QPU Training Subset ----
    print(f"\n[SUBSET] --- QPU Training Subset "
          f"(QPU_TRAIN_SUBSET={QPU_TRAIN_SUBSET}) ---")
    X_train, y_train = make_qpu_training_subset(
        X_train_full, y_train_full,
        subset_size=QPU_TRAIN_SUBSET,
        random_state=split * 42
    )
    n_train = X_train.shape[0]
    print(f"  Full train: {n_train_full} samples | "
          f"QPU train: {n_train} samples | "
          f"BQM vars: {n_train * N_BITS}")

    # ---- BQM Build ----
    print(f"\n[TIMING] --- BQM Build Phase (n_train={n_train}) ---")
    _t_build_start = time.perf_counter()
    bqm, K_train, scale, q_max, bit_weights, build_timing = build_dual_qsvm_bqm(
        X_train, y_train,
        n_bits=cfg["n_bits"],
        C=cfg["C"],
        balance_penalty=cfg["balance_penalty"],
        kernel=cfg["kernel"],
        rbf_gamma=cfg["rbf_gamma"]
    )
    t_build = time.perf_counter() - _t_build_start

    print(f"\n[TIMING] BQM Build Per-Sample Summary (n_train={n_train}):")
    print(f"  Total build time:             {t_build:.4f} s")
    print(f"  Per training sample (total):  {t_build/n_train*1e6:.3f} us")
    print(f"  Per training sample (kernel): "
          f"{build_timing['Kernel_sec']/n_train*1e6:.3f} us")
    print(f"  Per training sample (quad):   "
          f"{build_timing['Quad_sec']/n_train*1e6:.3f} us")
    print(f"  Per training sample (penalty):"
          f"{build_timing['Penalty_sec']/n_train*1e6:.3f} us")
    print(f"  Per sample-pair (quad):       "
          f"{build_timing['Quad_sec']/(n_train*n_train)*1e6:.4f} us")
    print(f"  BQM vars: {len(bqm.variables)}  "
          f"interactions: {len(bqm.quadratic)}")

    print("\nBQM created for QPU.")
    print("Number of BQM variables:", len(bqm.variables))
    print("Number of BQM interactions:", len(bqm.quadratic))
    print("q_max:", q_max)
    print("alpha scale:", scale)
    print("balance penalty:", cfg["balance_penalty"])

    # ---- QPU Solve ----
    print(f"\n[TIMING] --- QPU Solve Phase (n_train={n_train}) ---")
    _t_solve_start = time.perf_counter()
    sampleset, solver_used, chain_strength, solve_timing = solve_bqm_qpu(
        bqm=bqm,
        token=DWAVE_TOKEN,
        solver_name=QPU_SOLVER_NAME,
        num_reads=NUM_READS,
        annealing_time=ANNEALING_TIME,
        auto_scale=AUTO_SCALE,
        use_clique_first=USE_CLIQUE_SAMPLER_FIRST,
        label=f"QPU_QSVM_full_split{split}",
        n_train=n_train
    )
    t_solve = time.perf_counter() - _t_solve_start

    qpu_hw_sec = solve_timing.get("QPU_Time_sec", np.nan)
    queue_sec  = solve_timing.get("Queue_Network_Overhead_sec",
                                   solve_timing.get("Queue_Time_Est_sec", np.nan))
    wall_sec   = solve_timing.get("Wall_Clock_Submit_To_Result_sec",
                                   solve_timing.get("Wall_Clock_sec", np.nan))

    print("\nRaw best QPU BQM energy:", sampleset.first.energy)
    print(f"\n[TIMING] QPU Solve Per-Sample Summary (n_train={n_train}):")
    print(f"  Wall-clock submit->result:   {wall_sec:.4f} s")
    if not np.isnan(qpu_hw_sec):
        print(f"  QPU hardware time:           {qpu_hw_sec:.6f} s")
        print(f"  QPU hardware / n_train:      {qpu_hw_sec/n_train*1e6:.3f} us/sample")
        print(f"  QPU hardware / num_reads:    {qpu_hw_sec/NUM_READS*1e6:.3f} us/read")
    else:
        print(f"  QPU hardware time:           NOT RETURNED")
    if not np.isnan(queue_sec):
        print(f"  Queue+network overhead:      {queue_sec:.4f} s")
        print(f"  Queue+network / n_train:     {queue_sec/n_train*1e6:.3f} us/sample")

    # ---- Post-solve selection ----
    print(f"\n[TIMING] --- Post-Solve Selection Phase ---")
    best, candidates, selection_note, t_postsolve = select_best_qpu_solution(
        sampleset=sampleset,
        n_train=n_train,
        n_bits=cfg["n_bits"],
        scale=scale,
        bit_weights=bit_weights,
        y_train=y_train,
        K_train=K_train,
        balance_tolerance=1e-9
    )

    q_values      = best["q_values"]
    alpha         = best["alpha"]
    balance_value = best["balance_value"]
    best_energy   = best["energy"]
    true_obj      = best["true_svm_objective"]
    n_sv          = int(np.sum(alpha > 1e-8))

    print("\nSelection note:", selection_note)
    print("\nq values:", q_values)
    print("Alpha values:", alpha)
    print("SVM balance value sum(alpha_i * y_i):", balance_value)
    print("True SVM dual objective:", true_obj)
    print("Selected QPU BQM energy:", best_energy)
    print("Number of nonzero alpha/support vectors:", n_sv)
    print(f"  Selection note: {selection_note}")
    print(f"  Balance value:  {balance_value:.6f}  |  Nonzero alphas: {n_sv}")

    # ---- Bias ----
    print(f"\n[TIMING] --- Bias Computation ---")
    b, support_indices, t_bias = compute_bias_dual(
        alpha, y_train, K_train, C=cfg["C"]
    )
    print(f"\nBias b: {b:.6f}")
    print("Support vector indices:", support_indices)

    # ---- Weight extraction ----
    print(f"\n[TIMING] --- Weight Extraction ---")
    if cfg["kernel"] == "linear":
        w, t_weight = extract_linear_weight_from_alpha(
            alpha, X_train, y_train
        )
    else:
        w, t_weight = None, 0.0

    # ---- Training Set Prediction ----
    print(f"\n[TIMING] --- Training Set Prediction (n={n_train}) ---")
    _t_train_pred_start = time.perf_counter()
    y_prob_train, t_train_pred_total, t_train_kernel, t_train_score = \
        predict_probability_dual(
            X_train, X_train, alpha, y_train, b,
            kernel=cfg["kernel"], rbf_gamma=cfg["rbf_gamma"],
            label="TRAIN"
        )
    t_train_pred  = time.perf_counter() - _t_train_pred_start
    metrics_train = compute_metrics(y_train, y_prob_train)
    print(f"  Train Accuracy: {metrics_train['Accuracy']:.4f}  "
          f"ROC-AUC: {metrics_train['ROC-AUC']:.4f}")

    # ---- Test Set Prediction ----
    print(f"\n[TIMING] --- Test Set Prediction (n={n_test}) ---")
    _t_test_pred_start = time.perf_counter()
    y_prob_test, t_test_pred_total, t_test_kernel, t_test_score = \
        predict_probability_dual(
            X_test, X_train, alpha, y_train, b,
            kernel=cfg["kernel"], rbf_gamma=cfg["rbf_gamma"],
            label="TEST"
        )
    t_test_pred  = time.perf_counter() - _t_test_pred_start
    metrics_test = compute_metrics(y_test, y_prob_test)

    # ---- Metrics ----
    _t_metrics_start = time.perf_counter()
    metrics_test = compute_metrics(y_test, y_prob_test)
    t_metrics    = time.perf_counter() - _t_metrics_start

    t_run_total = time.perf_counter() - _t_split_start

    print(f"\n[RESULT] "
          f"Acc={metrics_test['Accuracy']:.4f} | "
          f"ROC-AUC={metrics_test['ROC-AUC']:.4f} | "
          f"F1={metrics_test['F1']:.4f} | "
          f"Precision={metrics_test['Precision']:.4f} | "
          f"Recall={metrics_test['Recall']:.4f} | "
          f"Sensitivity={metrics_test['Sensitivity']:.4f} | "
          f"Specificity={metrics_test['Specificity']:.4f} | "
          f"Kappa={metrics_test['Kappa']:.4f}")

    # ============================================================
    # PER-SAMPLE TIMING TABLE
    # ============================================================
    print("\n" + "=" * 70)
    print("[TIMING] ===== PER-SAMPLE TIMING TABLE =====")
    print(f"  n_train (QPU subset) = {n_train}  |  "
          f"n_train_full = {n_train_full}  |  "
          f"n_test = {n_test}  |  "
          f"n_support_vectors = {n_sv}")
    print("=" * 70)

    phase_times = {
        "Preprocessing (train)": {
            "total_sec": t_prep, "n": n_train_full,
            "denom_label": "train_sample"
        },
        "QPU Subset Selection": {
            "total_sec": 0.0, "n": n_train,
            "denom_label": "train_sample"
        },
        "BQM Build -- Total": {
            "total_sec": t_build, "n": n_train,
            "denom_label": "train_sample"
        },
        "BQM Build -- Kernel": {
            "total_sec": build_timing["Kernel_sec"], "n": n_train,
            "denom_label": "train_sample"
        },
        "BQM Build -- Quadratic": {
            "total_sec": build_timing["Quad_sec"], "n": n_train,
            "denom_label": "train_sample"
        },
        "BQM Build -- Penalty": {
            "total_sec": build_timing["Penalty_sec"], "n": n_train,
            "denom_label": "train_sample"
        },
        "QPU Solve -- Wall-clock": {
            "total_sec": wall_sec if not np.isnan(wall_sec) else t_solve,
            "n": n_train, "denom_label": "train_sample"
        },
        "Post-solve Selection": {
            "total_sec": t_postsolve, "n": n_train,
            "denom_label": "train_sample"
        },
        "Bias Computation": {
            "total_sec": t_bias, "n": max(n_sv, 1),
            "denom_label": "support_vector"
        },
        "Weight Extraction": {
            "total_sec": t_weight, "n": n_train,
            "denom_label": "train_sample"
        },
        "Train Inference -- Total": {
            "total_sec": t_train_pred, "n": n_train,
            "denom_label": "train_sample"
        },
        "Train Inference -- Kernel": {
            "total_sec": t_train_kernel, "n": n_train,
            "denom_label": "train_sample"
        },
        "Train Inference -- Score+Sigmoid": {
            "total_sec": t_train_score, "n": n_train,
            "denom_label": "train_sample"
        },
        "Preprocessing (test)": {
            "total_sec": t_prep * (n_test / (n_train_full + n_test)),
            "n": n_test, "denom_label": "test_sample"
        },
        "Test Inference -- Total": {
            "total_sec": t_test_pred, "n": n_test,
            "denom_label": "test_sample"
        },
        "Test Inference -- Kernel": {
            "total_sec": t_test_kernel, "n": n_test,
            "denom_label": "test_sample"
        },
        "Test Inference -- Score+Sigmoid": {
            "total_sec": t_test_score, "n": n_test,
            "denom_label": "test_sample"
        },
        "Metrics Computation": {
            "total_sec": t_metrics, "n": n_test,
            "denom_label": "test_sample"
        },
    }

    if not np.isnan(qpu_hw_sec):
        phase_times["QPU Hardware Access"] = {
            "total_sec": qpu_hw_sec, "n": n_train,
            "denom_label": "train_sample"
        }
    if not np.isnan(queue_sec):
        phase_times["QPU Queue+Network"] = {
            "total_sec": queue_sec, "n": n_train,
            "denom_label": "train_sample"
        }

    per_sample_df = per_sample_timing_table(
        phase_times, n_train, n_test, n_sv
    )
    display(per_sample_df)

    print("\n[TIMING] ===== KEY PER-SAMPLE HIGHLIGHTS =====")
    print(f"  BQM build per QPU train sample:     "
          f"{t_build/n_train*1e6:>10.3f} us  "
          f"({t_build:.4f} s total, n_qpu={n_train})")
    if not np.isnan(qpu_hw_sec):
        print(f"  QPU hardware per QPU train sample:  "
              f"{qpu_hw_sec/n_train*1e6:>10.3f} us  "
              f"({qpu_hw_sec:.6f} s total)")
        print(f"  QPU hardware per read:              "
              f"{qpu_hw_sec/NUM_READS*1e6:>10.3f} us  "
              f"({NUM_READS} reads)")
    print(f"  QPU wall-clock per QPU train sample:"
          f"{(wall_sec if not np.isnan(wall_sec) else t_solve)/n_train*1e6:>10.3f} us")
    print(f"  Train inference per train sample:   "
          f"{t_train_pred/n_train*1e6:>10.3f} us  "
          f"({t_train_pred:.6f} s total, n={n_train})")
    print(f"  Test  inference per test  sample:   "
          f"{t_test_pred/n_test*1e6:>10.3f} us  "
          f"({t_test_pred:.6f} s total, n={n_test})")
    print(f"  Metrics per test sample:            "
          f"{t_metrics/n_test*1e6:>10.3f} us")
    print(f"  Total wall-clock:                   "
          f"{t_run_total:.4f} s")

    # ---- CSV row ----
    row = {
        "Split":               split,
        "QPU_Solver":          QPU_SOLVER_NAME,
        "QPU_Sampler":         solver_used,
        "n_train_full":        n_train_full,
        "n_train_qpu":         n_train,
        "n_test":              n_test,
        "n_support_vectors":   n_sv,
        "Kernel":              cfg["kernel"],
        "C":                   cfg["C"],
        "n_bits":              cfg["n_bits"],
        "Balance_Penalty":     cfg["balance_penalty"],
        "BQM_Variables":       len(bqm.variables),
        "BQM_Interactions":    len(bqm.quadratic),
        "Num_Reads":           NUM_READS,
        "Annealing_Time_us":   ANNEALING_TIME,
        "Chain_Strength":      chain_strength,
        # Test metrics
        "Accuracy":            metrics_test["Accuracy"],
        "ROC_AUC":             metrics_test["ROC-AUC"],
        "PR_AUC":              metrics_test["PR-AUC"],
        "F1":                  metrics_test["F1"],
        "Precision":           metrics_test["Precision"],
        "Recall":              metrics_test["Recall"],
        "Sensitivity":         metrics_test["Sensitivity"],
        "Specificity":         metrics_test["Specificity"],
        "Kappa":               metrics_test["Kappa"],
        # Train metrics
        "Train_Accuracy":      metrics_train["Accuracy"],
        "Train_ROC_AUC":       metrics_train["ROC-AUC"],
        # SVM solution
        "Balance_Value":       balance_value,
        "Balance_Abs":         abs(balance_value),
        "Nonzero_Alpha_Count": n_sv,
        "Selection_Note":      selection_note,
        "Best_Energy":         best_energy,
        "True_SVM_Objective":  true_obj,
        # Total times
        "T_Split_sec":         round(t_split, 6),
        "T_Preprocess_sec":    round(t_prep, 6),
        "T_Build_BQM_sec":     round(t_build, 6),
        "T_Build_Kernel_sec":  round(build_timing["Kernel_sec"], 6),
        "T_Build_Quad_sec":    round(build_timing["Quad_sec"], 6),
        "T_Build_Penalty_sec": round(build_timing["Penalty_sec"], 6),
        "T_QPU_Solve_sec":     round(t_solve, 6),
        "T_QPU_Hardware_sec":
            round(qpu_hw_sec, 6) if not np.isnan(qpu_hw_sec) else np.nan,
        "T_QPU_Queue_sec":
            round(queue_sec, 6) if not np.isnan(queue_sec) else np.nan,
        "T_QPU_WallClock_sec": round(wall_sec, 6),
        "T_PostSolve_sec":     round(t_postsolve, 6),
        "T_Bias_sec":          round(t_bias, 6),
        "T_Weight_sec":        round(t_weight, 6),
        "T_Train_Pred_sec":    round(t_train_pred, 6),
        "T_Test_Pred_sec":     round(t_test_pred, 6),
        "T_Metrics_sec":       round(t_metrics, 6),
        "T_Total_sec":         round(t_run_total, 6),
        # Per QPU training sample (us)
        "PerTrainSample_Preprocess_us":
            round(t_prep/n_train_full*1e6, 4),
        "PerTrainSample_BuildBQM_us":
            round(t_build/n_train*1e6, 4),
        "PerTrainSample_BuildKernel_us":
            round(build_timing["Kernel_sec"]/n_train*1e6, 4),
        "PerTrainSample_BuildQuad_us":
            round(build_timing["Quad_sec"]/n_train*1e6, 4),
        "PerTrainSample_QPUWallClock_us":
            round((wall_sec if not np.isnan(wall_sec) else t_solve)
                  /n_train*1e6, 4),
        "PerTrainSample_QPUHardware_us":
            round(qpu_hw_sec/n_train*1e6, 4)
            if not np.isnan(qpu_hw_sec) else np.nan,
        "PerTrainSample_TrainInference_us":
            round(t_train_pred/n_train*1e6, 4),
        "PerRead_QPUHardware_us":
            round(qpu_hw_sec/NUM_READS*1e6, 4)
            if not np.isnan(qpu_hw_sec) else np.nan,
        # Per test sample (us)
        "PerTestSample_TestInference_us":
            round(t_test_pred/n_test*1e6, 4),
        "PerTestSample_InferenceKernel_us":
            round(t_test_kernel/n_test*1e6, 4),
        "PerTestSample_ScoreSigmoid_us":
            round(t_test_score/n_test*1e6, 4),
        "PerTestSample_Metrics_us":
            round(t_metrics/n_test*1e6, 4),
        # D-Wave raw timing fields
        "QPU_Access_us":
            solve_timing.get("DWave_qpu_access_time_us", np.nan),
        "QPU_Programming_us":
            solve_timing.get("DWave_qpu_programming_time_us", np.nan),
        "QPU_Anneal_per_read_us":
            solve_timing.get("QPU_Anneal_per_read_us", np.nan),
        "QPU_Readout_per_read_us":
            solve_timing.get("QPU_Readout_per_read_us", np.nan),
        "Embedding_Physical_Qubits":
            solve_timing.get("Embedding_Physical_Qubits", np.nan),
        "Embedding_Max_Chain":
            solve_timing.get("Embedding_Max_Chain", np.nan),
        "Embedding_Mean_Chain":
            solve_timing.get("Embedding_Mean_Chain", np.nan),
    }

    # Add all solve_timing fields for completeness
    for k, v in solve_timing.items():
        row[f"Timing_{k}"] = v

    summary.append(row)

    _t_csv_start = time.perf_counter()
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)
    t_csv = time.perf_counter() - _t_csv_start
    print(f"\n[TIMING] CSV saved: {t_csv:.4f} s  -> {AUTO_SAVE_PATH}")

    print("\n===== TIMING SUMMARY FOR THIS SPLIT =====")
    timing_display_df = (
        pd.DataFrame(
            [{"Stage": k, "Seconds_or_Value": v}
             for k, v in solve_timing.items()]
        )
        .sort_values("Stage")
        .reset_index(drop=True)
    )
    display(timing_display_df)

    if VISUALIZE:
        _t_vis_start = time.perf_counter()
        feature_importance_df = run_all_qpu_visualizations(
            bqm=bqm,
            sampleset=sampleset,
            K_train=K_train,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            y_prob_test=y_prob_test,
            y_prob_train=y_prob_train,
            alpha=alpha,
            w=w,
            b=b,
            metrics_test=metrics_test,
            metrics_train=metrics_train,
            selected_feature_names=selected_feature_names,
            scale=scale,
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"],
            solve_timing=solve_timing,
            n_train=n_train,
            num_reads=NUM_READS
        )
        _t_vis_end = time.perf_counter()
        print(f"  [TIMING] All visualizations: {_t_vis_end - _t_vis_start:.4f} s")

        if feature_importance_df is not None:
            print("\nFeature Importance Table:")
            display(feature_importance_df)

        # Per-sample timing plots
        plot_per_sample_timing(
            per_sample_df,
            title="BQM-QSVM Per-Sample Timing Breakdown"
        )
        plot_train_vs_test_per_sample(
            train_us=t_train_pred/n_train*1e6,
            test_us=t_test_pred/n_test*1e6,
            n_train=n_train,
            n_test=n_test
        )

    print(f"\n[TIMING] Total split {split} runtime: {t_run_total:.4f} s")


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_df = (
    pd.DataFrame(summary)
    .sort_values("Accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)

if VISUALIZE:
    plot_runtime_summary(summary_df)

_t_notebook_end = time.perf_counter()
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print(f"\n[TIMING] Total notebook runtime: "
      f"{_t_notebook_end - _t_notebook_start:.4f} s")
print("\nDONE.")

**<H1>BQM**

In [ ]:
# ============================================================
# Fully D-Wave Hybrid BQM-Based QSVM with Fixed Visualizations
# Dataset must already be loaded as: dff
# Target column: LUNG_CANCER
# Solver: D-Wave LeapHybridSampler
# ============================================================

!pip install -q --upgrade-strategy only-if-needed dwave-ocean-sdk dwave-neal matplotlib seaborn networkx

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import time
import os
from getpass import getpass

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score,
    roc_curve, precision_recall_curve
)

import dimod
from dimod import BinaryQuadraticModel
from dwave.system import LeapHybridSampler
from neal import SimulatedAnnealingSampler


# ============================================================
# D-WAVE TOKEN
# ============================================================

DWAVE_TOKEN = getpass("Paste your D-Wave API token: ")


# ============================================================
# USER CONTROLS
# ============================================================

SPLIT_MODE = "holdout"
N_RUNS = 1
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 100
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "live_bqm_qsvm_results.csv"
n_splits = 1

# BQM-QSVM settings
N_BITS = 1
DEFAULT_C = 0.3
KERNEL_TYPE = "linear"       # "linear" or "rbf"
RBF_GAMMA = 1.0

# Penalty for SVM balance constraint:
# sum(alpha_i * y_i) = 0
BALANCE_PENALTY = 5.0

# Solver mode:
# "hybrid_bqm" = D-Wave Hybrid BQM Solver
# "local_sa"   = Local Simulated Annealing
BQM_SOLVER_MODE = "hybrid_bqm"

# Local simulated annealing reads
NUM_READS_MODEL = 6000

# Hybrid BQM time
# None = use D-Wave minimum time automatically
# For stronger final run, try 30, 60, 120
MANUAL_BQM_TIME_LIMIT = None

# Visualization control
VISUALIZE = True


# ============================================================
# LOAD DATA
# dff must exist before running this cell
# ============================================================

if "dff" not in globals():
    raise ValueError("dff not found. Please load your dataset into a DataFrame named dff first.")

if TARGET not in dff.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dff.")

X_df = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()

y_raw = dff[TARGET].values
unique_targets = sorted(pd.Series(y_raw).dropna().unique())

if set(unique_targets) == {0, 1}:
    y = dff[TARGET].astype(int).values
else:
    if len(unique_targets) != 2:
        raise ValueError(f"Target must be binary. Found target values: {unique_targets}")

    target_map = {
        unique_targets[0]: 0,
        unique_targets[1]: 1
    }

    y = pd.Series(y_raw).map(target_map).astype(int).values
    print("\nTarget mapping used:")
    print(target_map)

X_raw = X_df.values.astype(float)

print(dff.info())


# ============================================================
# SELECTED FEATURES
# ============================================================

selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

max_index = max(selected_features_indices)

if max_index >= len(feature_names):
    raise ValueError(
        f"Selected feature index {max_index} is out of range. "
        f"Dataset has only {len(feature_names)} features excluding target."
    )

selected_feature_names = [feature_names[i] for i in selected_features_indices]
X_raw = X_raw[:, selected_features_indices]

print("\nSelected feature names:")
print(selected_feature_names)

print("\nTotal samples:", X_raw.shape[0])
print("Selected feature count:", X_raw.shape[1])


# ============================================================
# BASIC FUNCTIONS
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }


def compute_kernel_matrix(XA, XB=None, kernel="linear", rbf_gamma=1.0):
    if XB is None:
        XB = XA

    if kernel == "linear":
        K = XA @ XB.T

    elif kernel == "rbf":
        XA_sq = np.sum(XA ** 2, axis=1, keepdims=True)
        XB_sq = np.sum(XB ** 2, axis=1, keepdims=True).T
        dists = XA_sq + XB_sq - 2 * (XA @ XB.T)
        K = np.exp(-rbf_gamma * dists)

    else:
        raise ValueError("kernel must be 'linear' or 'rbf'.")

    return K


# ============================================================
# BUILD BQM DUAL QSVM
# ============================================================

def build_dual_qsvm_bqm(
    X,
    y,
    n_bits=3,
    C=0.3,
    balance_penalty=5.0,
    kernel="linear",
    rbf_gamma=1.0
):
    """
    SVM dual objective:

        Maximize:
            sum(alpha_i) - 1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij

        D-Wave minimizes:
            1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij - sum(alpha_i)

    Constraint:
        sum(alpha_i * y_i) = 0

    BQM penalty:
        balance_penalty * (sum(alpha_i * y_i))^2

    Binary alpha encoding:
        alpha_i = scale * (b_i0 + 2*b_i1 + 4*b_i2 + ...)

        scale = C / (2^n_bits - 1)
    """

    y_pm = (2 * y - 1).astype(float)
    n = X.shape[0]

    K = compute_kernel_matrix(X, kernel=kernel, rbf_gamma=rbf_gamma)

    q_max = (2 ** n_bits) - 1
    scale = C / q_max
    bit_weights = [2 ** k for k in range(n_bits)]

    bqm = BinaryQuadraticModel({}, {}, 0.0, dimod.BINARY)

    def bit_name(i, k):
        return f"a_{i}_{k}"

    def add_term(u, v, coef):
        if abs(coef) < 1e-12:
            return

        if u == v:
            bqm.add_linear(u, coef)
        else:
            bqm.add_quadratic(u, v, coef)

    # Add variables
    for i in range(n):
        for k in range(n_bits):
            bqm.add_variable(bit_name(i, k), 0.0)

    # Linear term: -sum(alpha_i)
    for i in range(n):
        for k in range(n_bits):
            u = bit_name(i, k)
            coeff_alpha = bit_weights[k] * scale
            bqm.add_linear(u, -coeff_alpha)

    # SVM quadratic term
    for i in range(n):
        for j in range(n):
            base_coef = 0.5 * y_pm[i] * y_pm[j] * K[i, j]

            for ki in range(n_bits):
                for kj in range(n_bits):
                    u = bit_name(i, ki)
                    v = bit_name(j, kj)

                    ci = bit_weights[ki] * scale
                    cj = bit_weights[kj] * scale

                    coef = base_coef * ci * cj
                    add_term(u, v, coef)

    # Balance penalty term
    for i in range(n):
        for j in range(n):
            base_coef = balance_penalty * y_pm[i] * y_pm[j]

            for ki in range(n_bits):
                for kj in range(n_bits):
                    u = bit_name(i, ki)
                    v = bit_name(j, kj)

                    ci = bit_weights[ki] * scale
                    cj = bit_weights[kj] * scale

                    coef = base_coef * ci * cj
                    add_term(u, v, coef)

    return bqm, K, scale, q_max, bit_weights


# ============================================================
# SOLVE BQM
# ============================================================

def solve_bqm_qsvm(
    bqm,
    token=None,
    solver_mode="hybrid_bqm",
    num_reads=6000,
    manual_time_limit=None,
    label="BQM-QSVM"
):
    if solver_mode == "local_sa":
        sampler = SimulatedAnnealingSampler()

        print("\nBQM solver: Local SimulatedAnnealingSampler")
        print("Using num_reads:", num_reads)

        sampleset = sampler.sample(
            bqm,
            num_reads=num_reads
        )

        used_time_limit = None
        solver_name = "SimulatedAnnealingSampler"

    elif solver_mode == "hybrid_bqm":
        if token is None or token.strip() == "":
            sampler = LeapHybridSampler()
        else:
            sampler = LeapHybridSampler(token=token)

        min_time = sampler.min_time_limit(bqm)

        if manual_time_limit is None:
            time_limit = max(3, int(np.ceil(min_time)))
        else:
            time_limit = max(3, int(np.ceil(min_time)), int(manual_time_limit))

        print("\nBQM solver:", sampler.solver.name)
        print("Minimum required time:", min_time)
        print("Using time_limit:", time_limit)

        sampleset = sampler.sample(
            bqm,
            time_limit=time_limit,
            label=label
        )

        used_time_limit = time_limit
        solver_name = sampler.solver.name

    else:
        raise ValueError("solver_mode must be 'hybrid_bqm' or 'local_sa'.")

    best_sample = sampleset.first.sample
    best_energy = sampleset.first.energy

    return best_sample, best_energy, sampleset, used_time_limit, solver_name


# ============================================================
# EXTRACT ALPHA, OBJECTIVE, WEIGHT, BIAS
# ============================================================

def extract_alpha_from_bqm_sample(sample, n, n_bits, scale, bit_weights):
    q_values = np.zeros(n)

    for i in range(n):
        q_i = 0

        for k in range(n_bits):
            q_i += bit_weights[k] * int(sample.get(f"a_{i}_{k}", 0))

        q_values[i] = q_i

    alpha = q_values * scale

    return q_values, alpha


def true_svm_dual_objective(alpha, y_train, K_train):
    y_pm = (2 * y_train - 1).astype(float)

    quad = 0.0

    for i in range(len(alpha)):
        for j in range(len(alpha)):
            quad += (
                0.5
                * alpha[i]
                * alpha[j]
                * y_pm[i]
                * y_pm[j]
                * K_train[i, j]
            )

    linear = -np.sum(alpha)

    return quad + linear


def select_best_bqm_solution(
    sampleset,
    n_train,
    n_bits,
    scale,
    bit_weights,
    y_train,
    K_train,
    balance_tolerance=1e-9
):
    y_pm = (2 * y_train - 1).astype(float)

    candidates = []

    for row in sampleset.data(["sample", "energy", "num_occurrences"]):
        sample = row.sample
        energy = row.energy
        num_occurrences = row.num_occurrences

        q_values, alpha = extract_alpha_from_bqm_sample(
            sample,
            n=n_train,
            n_bits=n_bits,
            scale=scale,
            bit_weights=bit_weights
        )

        balance_value = float(np.sum(alpha * y_pm))
        balance_abs = abs(balance_value)
        true_obj = true_svm_dual_objective(alpha, y_train, K_train)
        nonzero_alpha_count = int(np.sum(alpha > 1e-8))

        candidates.append({
            "sample": sample,
            "energy": energy,
            "num_occurrences": num_occurrences,
            "q_values": q_values,
            "alpha": alpha,
            "balance_value": balance_value,
            "balance_abs": balance_abs,
            "true_svm_objective": true_obj,
            "nonzero_alpha_count": nonzero_alpha_count
        })

    exact_balance = [
        c for c in candidates
        if c["balance_abs"] <= balance_tolerance and c["nonzero_alpha_count"] > 0
    ]

    if len(exact_balance) > 0:
        best = min(exact_balance, key=lambda c: c["true_svm_objective"])
        selection_note = "Exact-balanced BQM solution selected."
    else:
        best = min(candidates, key=lambda c: (c["balance_abs"], c["energy"]))
        selection_note = "No exact-balanced solution found. Selected smallest balance violation."

    return best, candidates, selection_note


def compute_bias_dual(alpha, y_train, K_train, C=0.3):
    y_pm = (2 * y_train - 1).astype(float)

    support_indices = np.where(alpha > 1e-8)[0]
    margin_indices = np.where((alpha > 1e-8) & (alpha < C - 1e-8))[0]

    if len(margin_indices) > 0:
        selected = margin_indices
    else:
        selected = support_indices

    if len(selected) == 0:
        return 0.0, support_indices

    b_values = []

    for i in selected:
        decision_without_b = np.sum(alpha * y_pm * K_train[i, :])
        b_i = y_pm[i] - decision_without_b
        b_values.append(b_i)

    b = float(np.mean(b_values))

    return b, support_indices


def decision_function_dual(
    X_query,
    X_train,
    alpha,
    y_train,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    y_pm = (2 * y_train - 1).astype(float)

    K_query = compute_kernel_matrix(
        X_query,
        X_train,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    scores = K_query @ (alpha * y_pm) + b

    return scores


def predict_probability_dual(
    X_query,
    X_train,
    alpha,
    y_train,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    scores = decision_function_dual(
        X_query,
        X_train,
        alpha,
        y_train,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    return sigmoid(scores)


def extract_linear_weight_from_alpha(alpha, X_train, y_train):
    y_pm = (2 * y_train - 1).astype(float)
    w = (alpha * y_pm) @ X_train
    return w


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_bqm_energy_convergence(sampleset):
    energies = np.asarray(sampleset.record.energy)
    best = np.minimum.accumulate(energies)

    plt.figure(figsize=(7, 4))
    plt.plot(best, marker="o", linewidth=1)
    plt.title("BQM/QUBO Energy Convergence")
    plt.xlabel("Returned Sample Index")
    plt.ylabel("Best Energy So Far")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_bqm_energy_hist(sampleset):
    energies = np.asarray(sampleset.record.energy)

    plt.figure(figsize=(7, 4))
    plt.hist(energies, bins=30)
    plt.title("BQM/QUBO Energy Distribution")
    plt.xlabel("Energy")
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_bqm_solution_frequency(sampleset, top_k=10):
    try:
        df_samples = sampleset.to_pandas_dataframe()
    except Exception as e:
        print("Cannot convert sampleset to dataframe:", e)
        return

    var_cols = [
        col for col in df_samples.columns
        if str(col).startswith("a_")
    ]

    if len(var_cols) == 0:
        print("No alpha bit variables found for solution frequency plot.")
        return

    if "num_occurrences" not in df_samples.columns:
        df_samples["num_occurrences"] = 1

    df_samples["_solution_signature"] = (
        df_samples[var_cols]
        .astype(int)
        .astype(str)
        .agg(",".join, axis=1)
    )

    counts = (
        df_samples
        .groupby("_solution_signature")["num_occurrences"]
        .sum()
        .sort_values(ascending=False)
        .head(top_k)
    )

    short_labels = [
        s[:80] + "..." if len(s) > 80 else s
        for s in counts.index
    ]

    plt.figure(figsize=(10, 5))
    plt.bar(range(len(counts)), counts.values)
    plt.xticks(range(len(counts)), short_labels, rotation=45, ha="right")
    plt.title(f"Top {top_k} BQM Solution Frequencies")
    plt.ylabel("Occurrences")
    plt.tight_layout()
    plt.show()


def plot_qubo_heatmap(bqm, max_variables=120):
    """
    FIXED VERSION:
    Does not use bqm.to_numpy_matrix(variable_order=subset),
    because dimod requires variable_order to contain all variables.
    This function manually builds a subset matrix.
    """

    all_vars = list(bqm.variables)

    if len(all_vars) > max_variables:
        var_order = all_vars[:max_variables]
        print(f"QUBO heatmap limited to first {max_variables} variables out of {len(all_vars)}.")
    else:
        var_order = all_vars

    var_index = {v: i for i, v in enumerate(var_order)}
    mat = np.zeros((len(var_order), len(var_order)))

    # Linear biases on diagonal
    for v in var_order:
        i = var_index[v]
        mat[i, i] = bqm.get_linear(v)

    # Quadratic biases
    for (u, v), bias in bqm.quadratic.items():
        if u in var_index and v in var_index:
            i = var_index[u]
            j = var_index[v]
            mat[i, j] = bias
            mat[j, i] = bias

    plt.figure(figsize=(8, 7))
    sns.heatmap(
        mat,
        cmap="coolwarm",
        center=0,
        xticklabels=False,
        yticklabels=False
    )
    plt.title("BQM/QUBO Matrix Heatmap")
    plt.xlabel("Binary Variables")
    plt.ylabel("Binary Variables")
    plt.show()


def plot_qubo_graph(bqm, max_edges=100):
    """
    Safe graph visualization.
    Shows only strongest quadratic interactions.
    """

    edges = []

    for (u, v), bias in bqm.quadratic.items():
        if abs(bias) > 0:
            edges.append((u, v, abs(bias)))

    if len(edges) == 0:
        print("No quadratic interactions found in BQM.")
        return

    edges = sorted(edges, key=lambda x: x[2], reverse=True)[:max_edges]

    H = nx.Graph()

    for u, v, weight in edges:
        H.add_edge(u, v, weight=weight)

    plt.figure(figsize=(9, 7))
    pos = nx.spring_layout(H, seed=42)

    nx.draw(
        H,
        pos,
        node_color="lightblue",
        edge_color="gray",
        with_labels=True,
        node_size=450,
        font_size=7
    )

    plt.title(f"BQM/QUBO Interaction Graph: Top {max_edges} Strongest Edges")
    plt.show()


def plot_kernel_matrix_heatmap(K, title="Training Kernel Matrix"):
    plt.figure(figsize=(7, 6))
    sns.heatmap(K, cmap="viridis")
    plt.title(title)
    plt.xlabel("Training Sample Index")
    plt.ylabel("Training Sample Index")
    plt.show()


def plot_dual_objective_matrix(K, y_train, scale):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    plt.figure(figsize=(7, 6))
    sns.heatmap(Q, cmap="coolwarm", center=0)
    plt.title("Dual QSVM Interaction Matrix")
    plt.xlabel("Training Sample j")
    plt.ylabel("Training Sample i")
    plt.show()


def plot_alpha_values(alpha):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(alpha)), alpha)
    plt.title("Optimized Alpha Values")
    plt.xlabel("Training Sample Index")
    plt.ylabel("Alpha")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_alpha_by_class(alpha, y_train):
    df_alpha = pd.DataFrame({
        "Alpha": alpha,
        "Class": y_train.astype(int)
    })

    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_alpha, x="Class", y="Alpha")
    sns.stripplot(data=df_alpha, x="Class", y="Alpha", alpha=0.7)
    plt.title("Alpha Distribution by Class")
    plt.xlabel("Class")
    plt.ylabel("Alpha")
    plt.show()


def plot_support_vectors_pca(X_train, y_train, alpha, title="Support Vectors in PCA Space"):
    support_idx = alpha > 1e-8

    if X_train.shape[1] > 2:
        X_vis = PCA(n_components=2).fit_transform(X_train)
    else:
        X_vis = X_train

    plt.figure(figsize=(7, 5))
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        s=50,
        label="Training Samples"
    )

    plt.scatter(
        X_vis[support_idx, 0],
        X_vis[support_idx, 1],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Support Vectors"
    )

    plt.title(title)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_weight_importance(w, selected_feature_names):
    importance = np.abs(w)

    df_imp = pd.DataFrame({
        "Feature": selected_feature_names,
        "Weight": w,
        "Abs_Weight": importance
    }).sort_values("Abs_Weight", ascending=False)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=df_imp, x="Abs_Weight", y="Feature")
    plt.title("Feature Importance from BQM-QSVM Linear Weight Vector")
    plt.xlabel("|Weight|")
    plt.ylabel("Feature")
    plt.show()

    return df_imp


def plot_confusion_matrix_custom(y_true, y_prob, threshold=0.5, title="Confusion Matrix"):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def plot_roc_curve_custom(y_true, y_prob):
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_score = roc_auc_score(y_true, y_prob)
    except Exception as e:
        print("ROC curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_pr_curve_custom(y_true, y_prob):
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap_score = average_precision_score(y_true, y_prob)
    except Exception as e:
        print("PR curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC/AP = {ap_score:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_prediction_distribution(y_true, y_prob):
    df_pred = pd.DataFrame({
        "Probability": y_prob,
        "Class": y_true.astype(int)
    })

    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_pred, x="Probability", hue="Class", bins=30, kde=True)
    plt.title("Predicted Probability Distribution")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.show()


def plot_metrics_bar(metrics):
    selected_metrics = {
        "Accuracy": metrics.get("Accuracy", np.nan),
        "Precision": metrics.get("Precision", np.nan),
        "Recall": metrics.get("Recall", np.nan),
        "F1": metrics.get("F1", np.nan),
        "ROC-AUC": metrics.get("ROC-AUC", np.nan),
        "PR-AUC": metrics.get("PR-AUC", np.nan),
        "Sensitivity": metrics.get("Sensitivity", np.nan),
        "Specificity": metrics.get("Specificity", np.nan),
        "Kappa": metrics.get("Kappa", np.nan)
    }

    names = list(selected_metrics.keys())
    values = list(selected_metrics.values())

    plt.figure(figsize=(10, 4))
    plt.bar(names, values)
    plt.ylim(0, 1.05)
    plt.title("Model Performance Metrics")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_decision_boundary_pca_dual(
    X_train,
    y_train,
    alpha,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    if X_train.shape[1] > 2:
        pca = PCA(n_components=2)
        X_vis = pca.fit_transform(X_train)
    else:
        pca = None
        X_vis = X_train

    x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
    y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid_2d = np.c_[xx.ravel(), yy.ravel()]

    if pca is not None:
        grid_full = pca.inverse_transform(grid_2d)
    else:
        grid_full = grid_2d

    scores = decision_function_dual(
        grid_full,
        X_train,
        alpha,
        y_train,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    z = sigmoid(scores)
    z = z.reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, z, levels=50, alpha=0.8)
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        edgecolors="black",
        s=45
    )
    plt.title("Decision Boundary in PCA Space")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(label="Predicted Probability")
    plt.show()


def plot_runtime_summary(summary_df):
    if "Runtime_sec" not in summary_df.columns:
        print("Runtime_sec column not found.")
        return

    plt.figure(figsize=(8, 4))
    sns.barplot(data=summary_df, x="Model", y="Runtime_sec")
    plt.title("Runtime Comparison")
    plt.xlabel("Model")
    plt.ylabel("Runtime Second")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def run_all_bqm_visualizations(
    bqm,
    sampleset,
    K_train,
    X_train,
    y_train,
    X_test,
    y_test,
    y_prob,
    alpha,
    w,
    b,
    metrics,
    selected_feature_names,
    scale,
    kernel,
    rbf_gamma
):
    print("\n========== BQM / Solver Visualizations ==========")
    plot_bqm_energy_convergence(sampleset)
    plot_bqm_energy_hist(sampleset)
    plot_bqm_solution_frequency(sampleset, top_k=10)

    print("\n========== QUBO Structure Visualizations ==========")
    plot_qubo_heatmap(bqm, max_variables=120)
    plot_qubo_graph(bqm, max_edges=100)

    print("\n========== Kernel / Objective Visualizations ==========")
    plot_kernel_matrix_heatmap(K_train, title="Training Kernel Matrix")
    plot_dual_objective_matrix(K_train, y_train, scale)

    print("\n========== Alpha / Support Vector Visualizations ==========")
    plot_alpha_values(alpha)
    plot_alpha_by_class(alpha, y_train)
    plot_support_vectors_pca(X_train, y_train, alpha)

    print("\n========== Model Performance Visualizations ==========")
    plot_confusion_matrix_custom(y_test, y_prob, threshold=0.5)
    plot_roc_curve_custom(y_test, y_prob)
    plot_pr_curve_custom(y_test, y_prob)
    plot_prediction_distribution(y_test, y_prob)
    plot_metrics_bar(metrics)

    print("\n========== Feature / Decision Boundary Visualizations ==========")

    feature_importance_df = None

    if kernel == "linear" and w is not None:
        feature_importance_df = plot_weight_importance(w, selected_feature_names)
    else:
        print("Feature weight importance is only directly available for linear kernel.")

    plot_decision_boundary_pca_dual(
        X_train,
        y_train,
        alpha,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    return feature_importance_df


# ============================================================
# MAIN LOOP
# ============================================================

summary = []

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

dual_configs = {
    "DW_BQM_QSVM_SoftMargin_LowC": dict(
        kernel=KERNEL_TYPE,
        C=DEFAULT_C,
        n_bits=N_BITS,
        rbf_gamma=RBF_GAMMA,
        balance_penalty=BALANCE_PENALTY
    )
}

total_models = len(dual_configs) * n_splits
completed_jobs = 0

for split in range(1, n_splits + 1):

    print("\n============================================================")
    print(f"Split {split}/{n_splits}")
    print("============================================================")

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * 42
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    print("Train shape:", X_train.shape)
    print("Test shape :", X_test.shape)

    for name, cfg in dual_configs.items():

        start = time.time()

        print("\n------------------------------------------------------------")
        print("Running model:", name)
        print("Config:", cfg)
        print("------------------------------------------------------------")

        bqm, K_train, scale, q_max, bit_weights = build_dual_qsvm_bqm(
            X_train,
            y_train,
            n_bits=cfg["n_bits"],
            C=cfg["C"],
            balance_penalty=cfg["balance_penalty"],
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"]
        )

        print("\nBQM created.")
        print("Number of BQM variables:", len(bqm.variables))
        print("Number of BQM interactions:", len(bqm.quadratic))
        print("q_max:", q_max)
        print("alpha scale:", scale)
        print("balance penalty:", cfg["balance_penalty"])

        raw_best_sample, raw_best_energy, sampleset, used_time_limit, solver_name = solve_bqm_qsvm(
            bqm,
            token=DWAVE_TOKEN,
            solver_mode=BQM_SOLVER_MODE,
            num_reads=NUM_READS_MODEL,
            manual_time_limit=MANUAL_BQM_TIME_LIMIT,
            label=f"{name}_split_{split}"
        )

        print("\nRaw best BQM energy:", raw_best_energy)

        n_train = X_train.shape[0]

        best_candidate, candidates, selection_note = select_best_bqm_solution(
            sampleset=sampleset,
            n_train=n_train,
            n_bits=cfg["n_bits"],
            scale=scale,
            bit_weights=bit_weights,
            y_train=y_train,
            K_train=K_train,
            balance_tolerance=1e-9
        )

        print("\nSelection note:")
        print(selection_note)

        q_values = best_candidate["q_values"]
        alpha = best_candidate["alpha"]
        balance_value = best_candidate["balance_value"]
        best_energy = best_candidate["energy"]
        true_obj = best_candidate["true_svm_objective"]

        print("\nq values:")
        print(q_values)

        print("\nAlpha values:")
        print(alpha)

        print("\nSVM balance value sum(alpha_i * y_i):")
        print(balance_value)

        print("\nTrue SVM dual objective without penalty:")
        print(true_obj)

        print("\nSelected BQM energy:")
        print(best_energy)

        print("\nNumber of nonzero alpha/support vectors:")
        print(np.sum(alpha > 1e-8))

        b, support_indices = compute_bias_dual(
            alpha,
            y_train,
            K_train,
            C=cfg["C"]
        )

        print("\nBias b:", b)
        print("Support vector indices:", support_indices)

        if cfg["kernel"] == "linear":
            w = extract_linear_weight_from_alpha(alpha, X_train, y_train)
        else:
            w = None

        y_prob = predict_probability_dual(
            X_test,
            X_train,
            alpha,
            y_train,
            b,
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"]
        )

        metrics = compute_metrics(y_test, y_prob)

        runtime = time.time() - start
        completed_jobs += 1
        progress = (completed_jobs / total_models) * 100

        print(
            f"\n[{progress:6.2f}%] "
            f"{name} | "
            f"Acc={metrics['Accuracy']:.4f} | "
            f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
            f"PR-AUC={metrics['PR-AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Precision={metrics['Precision']:.4f} | "
            f"Recall={metrics['Recall']:.4f} | "
            f"Sensitivity={metrics['Sensitivity']:.4f} | "
            f"Specificity={metrics['Specificity']:.4f} | "
            f"Kappa={metrics['Kappa']:.4f} | "
            f"Runtime={runtime:.2f}s | "
            f"Solver={solver_name} | "
            f"BQM_time_limit={used_time_limit}"
        )

        row = {
            "Model": name,
            "Split": split,
            "Solver_Mode": BQM_SOLVER_MODE,
            "Solver_Name": solver_name,
            "Kernel": cfg["kernel"],
            "C": cfg["C"],
            "n_bits": cfg["n_bits"],
            "RBF_Gamma": cfg["rbf_gamma"],
            "Balance_Penalty": cfg["balance_penalty"],
            "Train_Size": X_train.shape[0],
            "Test_Size": X_test.shape[0],
            "Selected_Features": selected_feature_names,
            "BQM_Variables": len(bqm.variables),
            "BQM_Interactions": len(bqm.quadratic),
            "Accuracy_mean": metrics["Accuracy"],
            "ROC_AUC_mean": metrics["ROC-AUC"],
            "PR_AUC": metrics["PR-AUC"],
            "F1_mean": metrics["F1"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "Kappa": metrics["Kappa"],
            "Runtime_sec": runtime,
            "BQM_time_limit_sec": used_time_limit,
            "Raw_Best_Energy": raw_best_energy,
            "Selected_Best_Energy": best_energy,
            "True_SVM_Objective": true_obj,
            "Balance_Value": balance_value,
            "Balance_Abs": abs(balance_value),
            "Nonzero_Alpha_Count": int(np.sum(alpha > 1e-8)),
            "Selection_Note": selection_note
        }

        summary.append(row)

        pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

        if VISUALIZE:
            feature_importance_df = run_all_bqm_visualizations(
                bqm=bqm,
                sampleset=sampleset,
                K_train=K_train,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                y_prob=y_prob,
                alpha=alpha,
                w=w,
                b=b,
                metrics=metrics,
                selected_feature_names=selected_feature_names,
                scale=scale,
                kernel=cfg["kernel"],
                rbf_gamma=cfg["rbf_gamma"]
            )

            if feature_importance_df is not None:
                print("\nFeature Importance Table:")
                display(feature_importance_df)


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_df = (
    pd.DataFrame(summary)
    .sort_values("Accuracy_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)

if VISUALIZE:
    plot_runtime_summary(summary_df)

print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>CQM**

In [ ]:
# ============================================================
# Fully D-Wave Hybrid CQM-Based QSVM with Visualizations
# Dataset must already be loaded as: dff
# Target column: LUNG_CANCER
# Solver: D-Wave LeapHybridCQMSampler
# ============================================================

!pip install -q --upgrade-strategy only-if-needed dwave-ocean-sdk matplotlib seaborn networkx

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import time
import os
from getpass import getpass

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score,
    roc_curve, precision_recall_curve
)

import dimod
from dwave.system import LeapHybridCQMSampler


# ============================================================
# D-WAVE TOKEN
# ============================================================

DWAVE_TOKEN = getpass("Paste your D-Wave API token: ")


# ============================================================
# USER CONTROLS
# ============================================================

SPLIT_MODE = "holdout"
N_RUNS = 1
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 100
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "live_cqm_qsvm_results.csv"
n_splits = 1

# CQM-QSVM settings
N_BITS = 1
DEFAULT_C = 0.3
KERNEL_TYPE = "linear"      # "linear" or "rbf"
RBF_GAMMA = 1.0

# CQM solver time
# None = automatically use minimum time required by D-Wave.
# For more stable results, try 30, 60, 120.
MANUAL_CQM_TIME_LIMIT = None

# Visualization control
VISUALIZE = True


# ============================================================
# LOAD DATA
# dff must exist before running this cell
# ============================================================

if "dff" not in globals():
    raise ValueError("dff not found. Please load your dataset into a DataFrame named dff first.")

if TARGET not in dff.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dff.")

X_df = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()
y = dff[TARGET].astype(int).values
X_raw = X_df.values.astype(float)

print(dff.info())


# ============================================================
# SELECTED FEATURES
# ============================================================

selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

max_index = max(selected_features_indices)

if max_index >= len(feature_names):
    raise ValueError(
        f"Selected feature index {max_index} is out of range. "
        f"Dataset has only {len(feature_names)} features excluding target."
    )

selected_feature_names = [feature_names[i] for i in selected_features_indices]
X_raw = X_raw[:, selected_features_indices]

print("\nSelected feature names:")
print(selected_feature_names)

print("\nTotal samples:", X_raw.shape[0])
print("Selected feature count:", X_raw.shape[1])


# ============================================================
# BASIC FUNCTIONS
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def predict(X, w, b=0.0):
    return sigmoid(X @ w + b)


def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }


def compute_kernel_matrix(X, kernel="linear", rbf_gamma=1.0):
    if kernel == "linear":
        K = X @ X.T

    elif kernel == "rbf":
        sq = np.sum(X ** 2, axis=1, keepdims=True)
        dists = sq + sq.T - 2 * (X @ X.T)
        K = np.exp(-rbf_gamma * dists)

    else:
        raise ValueError("kernel must be 'linear' or 'rbf'.")

    return K


# ============================================================
# BUILD CQM DUAL QSVM
# ============================================================

def build_dual_qsvm_cqm(
    X,
    y,
    n_bits=3,
    C=0.3,
    kernel="linear",
    rbf_gamma=1.0,
    add_nonzero_alpha_constraint=True
):
    """
    SVM dual objective:

        Maximize:
            sum(alpha_i) - 1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij

        D-Wave minimizes:
            1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij - sum(alpha_i)

    Constraint:
        sum(alpha_i * y_i) = 0

    CQM discretization:
        alpha_i = q_i * scale
        q_i ∈ {0, 1, ..., 2^n_bits - 1}
        scale = C / (2^n_bits - 1)
    """

    y_pm = (2 * y - 1).astype(int)
    n = X.shape[0]

    K = compute_kernel_matrix(X, kernel=kernel, rbf_gamma=rbf_gamma)

    q_max = (2 ** n_bits) - 1
    scale = C / q_max

    cqm = dimod.ConstrainedQuadraticModel()

    q_vars = [
        dimod.Integer(f"q_{i}", lower_bound=0, upper_bound=q_max)
        for i in range(n)
    ]

    objective = 0

    # Quadratic objective:
    # 1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij
    for i in range(n):
        for j in range(i, n):

            if i == j:
                coef = 0.5 * y_pm[i] * y_pm[j] * K[i, j] * (scale ** 2)
            else:
                coef = y_pm[i] * y_pm[j] * K[i, j] * (scale ** 2)

            objective += float(coef) * q_vars[i] * q_vars[j]

    # Linear objective:
    # -sum(alpha_i)
    for i in range(n):
        objective += float(-scale) * q_vars[i]

    cqm.set_objective(objective)

    # SVM balance constraint:
    # sum(alpha_i * y_i) = 0
    # Since alpha_i = q_i * scale:
    # sum(q_i * y_i) = 0
    balance_constraint = 0

    for i in range(n):
        balance_constraint += int(y_pm[i]) * q_vars[i]

    cqm.add_constraint(balance_constraint == 0, label="svm_balance_constraint")

    # Prevent all-zero alpha solution
    if add_nonzero_alpha_constraint:
        total_q = 0

        for i in range(n):
            total_q += q_vars[i]

        cqm.add_constraint(total_q >= 2, label="nonzero_alpha_constraint")

    return cqm, K, scale, q_max


# ============================================================
# SOLVE CQM
# ============================================================

def solve_cqm_qsvm(cqm, token=None, manual_time_limit=None, label="CQM-QSVM"):
    if token is None or token.strip() == "":
        sampler = LeapHybridCQMSampler()
    else:
        sampler = LeapHybridCQMSampler(token=token)

    min_time = sampler.min_time_limit(cqm)

    if manual_time_limit is None:
        time_limit = max(5, int(np.ceil(min_time)))
    else:
        time_limit = max(5, int(np.ceil(min_time)), int(manual_time_limit))

    print("\nCQM solver:", sampler.solver.name)
    print("Minimum required time:", min_time)
    print("Using time_limit:", time_limit)

    sampleset = sampler.sample_cqm(
        cqm,
        time_limit=time_limit,
        label=label
    )

    feasible_sampleset = sampleset.filter(lambda row: row.is_feasible)

    if len(feasible_sampleset) > 0:
        best_sample = feasible_sampleset.first.sample
        best_energy = feasible_sampleset.first.energy
        feasible_found = True
    else:
        best_sample = sampleset.first.sample
        best_energy = sampleset.first.energy
        feasible_found = False

    return best_sample, best_energy, feasible_found, sampleset, time_limit


# ============================================================
# EXTRACT ALPHA, WEIGHT, BIAS
# ============================================================

def extract_alpha_from_cqm_sample(sample, n, scale):
    q_values = np.zeros(n)

    for i in range(n):
        q_values[i] = sample.get(f"q_{i}", 0)

    alpha = q_values * scale

    return q_values, alpha


def extract_dual_weights_from_alpha(alpha, X, y):
    y_pm = (2 * y - 1).astype(float)
    w = (alpha * y_pm) @ X
    return w


def compute_bias_from_support_vectors(alpha, X, y, w, C=0.3):
    y_pm = (2 * y - 1).astype(float)

    support_indices = np.where(alpha > 1e-8)[0]
    margin_indices = np.where((alpha > 1e-8) & (alpha < C - 1e-8))[0]

    if len(margin_indices) > 0:
        selected = margin_indices
    else:
        selected = support_indices

    if len(selected) == 0:
        return 0.0, support_indices

    b_values = []

    for i in selected:
        b_i = y_pm[i] - np.dot(X[i], w)
        b_values.append(b_i)

    b = float(np.mean(b_values))

    return b, support_indices


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_cqm_energy_convergence(sampleset):
    energies = np.asarray(sampleset.record.energy)
    best = np.minimum.accumulate(energies)

    plt.figure(figsize=(7, 4))
    plt.plot(best, marker="o", linewidth=1)
    plt.title("CQM Energy Convergence")
    plt.xlabel("Returned Sample Index")
    plt.ylabel("Best Energy So Far")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_cqm_energy_hist(sampleset):
    energies = np.asarray(sampleset.record.energy)

    plt.figure(figsize=(7, 4))
    plt.hist(energies, bins=30)
    plt.title("CQM Energy Distribution")
    plt.xlabel("Energy")
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_cqm_feasibility(sampleset):
    feasible_flags = []

    try:
        for row in sampleset.data(["is_feasible"]):
            feasible_flags.append(row.is_feasible)
    except Exception:
        print("Feasibility data not available.")
        return

    counts = [
        np.sum(np.array(feasible_flags) == False),
        np.sum(np.array(feasible_flags) == True)
    ]

    plt.figure(figsize=(5, 4))
    plt.bar(["Infeasible", "Feasible"], counts)
    plt.title("CQM Feasible vs Infeasible Solutions")
    plt.ylabel("Number of Solutions")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_feasible_energy_distribution(sampleset):
    energies = []
    feasible_flags = []

    try:
        for row in sampleset.data(["energy", "is_feasible"]):
            energies.append(row.energy)
            feasible_flags.append(row.is_feasible)
    except Exception:
        print("Feasible energy information not available.")
        return

    df_plot = pd.DataFrame({
        "Energy": energies,
        "Feasible": feasible_flags
    })

    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_plot, x="Energy", hue="Feasible", bins=30, kde=False)
    plt.title("Energy Distribution by Feasibility")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_cqm_solution_frequency(sampleset, top_k=10):
    try:
        df_samples = sampleset.to_pandas_dataframe()
    except Exception as e:
        print("Cannot convert sampleset to dataframe:", e)
        return

    var_cols = [
        col for col in df_samples.columns
        if str(col).startswith("q_")
    ]

    if len(var_cols) == 0:
        print("No q_i variables found for solution frequency plot.")
        return

    if "num_occurrences" not in df_samples.columns:
        df_samples["num_occurrences"] = 1

    df_samples["_solution_signature"] = (
        df_samples[var_cols]
        .astype(int)
        .astype(str)
        .agg(",".join, axis=1)
    )

    counts = (
        df_samples
        .groupby("_solution_signature")["num_occurrences"]
        .sum()
        .sort_values(ascending=False)
        .head(top_k)
    )

    short_labels = [
        s[:80] + "..." if len(s) > 80 else s
        for s in counts.index
    ]

    plt.figure(figsize=(10, 5))
    plt.bar(range(len(counts)), counts.values)
    plt.xticks(range(len(counts)), short_labels, rotation=45, ha="right")
    plt.title(f"Top {top_k} CQM Solution Frequencies")
    plt.ylabel("Occurrences")
    plt.tight_layout()
    plt.show()


def plot_kernel_matrix_heatmap(K, title="Training Kernel Matrix"):
    plt.figure(figsize=(7, 6))
    sns.heatmap(K, cmap="viridis")
    plt.title(title)
    plt.xlabel("Training Sample Index")
    plt.ylabel("Training Sample Index")
    plt.show()


def plot_dual_objective_matrix(K, y_train, scale):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    plt.figure(figsize=(7, 6))
    sns.heatmap(Q, cmap="coolwarm", center=0)
    plt.title("CQM Dual Objective Interaction Matrix")
    plt.xlabel("q_j")
    plt.ylabel("q_i")
    plt.show()


def plot_dual_interaction_graph(K, y_train, scale, max_edges=80):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    n = Q.shape[0]
    G = nx.Graph()

    for i in range(n):
        G.add_node(f"q_{i}")

    edges = []

    for i in range(n):
        for j in range(i + 1, n):
            weight = abs(Q[i, j])
            if weight > 0:
                edges.append((i, j, weight))

    edges = sorted(edges, key=lambda x: x[2], reverse=True)[:max_edges]

    for i, j, weight in edges:
        G.add_edge(f"q_{i}", f"q_{j}", weight=weight)

    plt.figure(figsize=(9, 7))
    pos = nx.spring_layout(G, seed=42)

    nx.draw(
        G,
        pos,
        with_labels=True,
        node_size=450,
        font_size=8,
        edge_color="gray"
    )

    plt.title(f"CQM Dual Interaction Graph: Top {max_edges} Edges")
    plt.show()


def plot_alpha_values(alpha):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(alpha)), alpha)
    plt.title("Optimized Alpha Values")
    plt.xlabel("Training Sample Index")
    plt.ylabel("Alpha")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_alpha_by_class(alpha, y_train):
    df_alpha = pd.DataFrame({
        "Alpha": alpha,
        "Class": y_train.astype(int)
    })

    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_alpha, x="Class", y="Alpha")
    sns.stripplot(data=df_alpha, x="Class", y="Alpha", alpha=0.7)
    plt.title("Alpha Distribution by Class")
    plt.xlabel("Class")
    plt.ylabel("Alpha")
    plt.show()


def plot_support_vectors_pca(X_train, y_train, alpha, title="Support Vectors in PCA Space"):
    support_idx = alpha > 1e-8

    if X_train.shape[1] > 2:
        X_vis = PCA(n_components=2).fit_transform(X_train)
    else:
        X_vis = X_train

    plt.figure(figsize=(7, 5))
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        s=50,
        label="Training Samples"
    )

    plt.scatter(
        X_vis[support_idx, 0],
        X_vis[support_idx, 1],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Support Vectors"
    )

    plt.title(title)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_weight_importance(w, selected_feature_names):
    importance = np.abs(w)

    df_imp = pd.DataFrame({
        "Feature": selected_feature_names,
        "Weight": w,
        "Abs_Weight": importance
    }).sort_values("Abs_Weight", ascending=False)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=df_imp, x="Abs_Weight", y="Feature")
    plt.title("Feature Importance from CQM-QSVM Weight Vector")
    plt.xlabel("|Weight|")
    plt.ylabel("Feature")
    plt.show()

    return df_imp


def plot_confusion_matrix_custom(y_true, y_prob, threshold=0.5, title="Confusion Matrix"):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def plot_roc_curve_custom(y_true, y_prob):
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_score = roc_auc_score(y_true, y_prob)
    except Exception as e:
        print("ROC curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_pr_curve_custom(y_true, y_prob):
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap_score = average_precision_score(y_true, y_prob)
    except Exception as e:
        print("PR curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC/AP = {ap_score:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_prediction_distribution(y_true, y_prob):
    df_pred = pd.DataFrame({
        "Probability": y_prob,
        "Class": y_true.astype(int)
    })

    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_pred, x="Probability", hue="Class", bins=30, kde=True)
    plt.title("Predicted Probability Distribution")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.show()


def plot_metrics_bar(metrics):
    selected_metrics = {
        "Accuracy": metrics.get("Accuracy", np.nan),
        "Precision": metrics.get("Precision", np.nan),
        "Recall": metrics.get("Recall", np.nan),
        "F1": metrics.get("F1", np.nan),
        "ROC-AUC": metrics.get("ROC-AUC", np.nan),
        "PR-AUC": metrics.get("PR-AUC", np.nan),
        "Sensitivity": metrics.get("Sensitivity", np.nan),
        "Specificity": metrics.get("Specificity", np.nan),
        "Kappa": metrics.get("Kappa", np.nan)
    }

    names = list(selected_metrics.keys())
    values = list(selected_metrics.values())

    plt.figure(figsize=(10, 4))
    plt.bar(names, values)
    plt.ylim(0, 1.05)
    plt.title("Model Performance Metrics")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_decision_boundary_pca(X_train, y_train, w, b=0.0):
    if X_train.shape[1] > 2:
        pca = PCA(n_components=2)
        X_vis = pca.fit_transform(X_train)
    else:
        pca = None
        X_vis = X_train

    x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
    y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid_2d = np.c_[xx.ravel(), yy.ravel()]

    if pca is not None:
        grid_full = pca.inverse_transform(grid_2d)
    else:
        grid_full = grid_2d

    z = sigmoid(grid_full @ w + b)
    z = z.reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, z, levels=50, alpha=0.8)
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        edgecolors="black",
        s=45
    )
    plt.title("Decision Boundary in PCA Space")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(label="Predicted Probability")
    plt.show()


def plot_runtime_summary(summary_df):
    if "Runtime_sec" not in summary_df.columns:
        print("Runtime_sec column not found.")
        return

    plt.figure(figsize=(8, 4))
    sns.barplot(data=summary_df, x="Model", y="Runtime_sec")
    plt.title("Runtime Comparison")
    plt.xlabel("Model")
    plt.ylabel("Runtime Second")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def run_all_cqm_visualizations(
    sampleset,
    K_train,
    X_train,
    y_train,
    X_test,
    y_test,
    y_prob,
    alpha,
    w,
    b,
    metrics,
    selected_feature_names,
    scale
):
    print("\n========== CQM / Solver Visualizations ==========")
    plot_cqm_energy_convergence(sampleset)
    plot_cqm_energy_hist(sampleset)
    plot_cqm_feasibility(sampleset)
    plot_feasible_energy_distribution(sampleset)
    plot_cqm_solution_frequency(sampleset, top_k=10)

    print("\n========== Kernel / Objective Visualizations ==========")
    plot_kernel_matrix_heatmap(K_train, title="Training Kernel Matrix")
    plot_dual_objective_matrix(K_train, y_train, scale)
    plot_dual_interaction_graph(K_train, y_train, scale, max_edges=80)

    print("\n========== Alpha / Support Vector Visualizations ==========")
    plot_alpha_values(alpha)
    plot_alpha_by_class(alpha, y_train)
    plot_support_vectors_pca(X_train, y_train, alpha)

    print("\n========== Model Performance Visualizations ==========")
    plot_confusion_matrix_custom(y_test, y_prob, threshold=0.5)
    plot_roc_curve_custom(y_test, y_prob)
    plot_pr_curve_custom(y_test, y_prob)
    plot_prediction_distribution(y_test, y_prob)
    plot_metrics_bar(metrics)

    print("\n========== Feature / Decision Boundary Visualizations ==========")
    feature_importance_df = plot_weight_importance(w, selected_feature_names)
    plot_decision_boundary_pca(X_train, y_train, w, b)

    return feature_importance_df


# ============================================================
# MAIN LOOP
# ============================================================

summary = []

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

dual_configs = {
    "DW_CQM_QSVM_SoftMargin_LowC": dict(
        kernel=KERNEL_TYPE,
        C=DEFAULT_C,
        n_bits=N_BITS,
        rbf_gamma=RBF_GAMMA
    )
}

total_models = len(dual_configs) * n_splits
completed_jobs = 0

for split in range(1, n_splits + 1):

    print("\n============================================================")
    print(f"Split {split}/{n_splits}")
    print("============================================================")

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * 42
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    print("Train shape:", X_train.shape)
    print("Test shape :", X_test.shape)

    for name, cfg in dual_configs.items():

        start = time.time()

        print("\n------------------------------------------------------------")
        print("Running model:", name)
        print("Config:", cfg)
        print("------------------------------------------------------------")

        cqm, K_train, scale, q_max = build_dual_qsvm_cqm(
            X_train,
            y_train,
            n_bits=cfg["n_bits"],
            C=cfg["C"],
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"],
            add_nonzero_alpha_constraint=True
        )

        print("\nCQM created.")
        print("Number of CQM variables:", len(cqm.variables))
        print("Number of CQM constraints:", len(cqm.constraints))
        print("q_max:", q_max)
        print("alpha scale:", scale)

        best_sample, best_energy, feasible_found, sampleset, used_time_limit = solve_cqm_qsvm(
            cqm,
            token=DWAVE_TOKEN,
            manual_time_limit=MANUAL_CQM_TIME_LIMIT,
            label=f"{name}_split_{split}"
        )

        print("\nFeasible solution found:", feasible_found)
        print("Best energy:", best_energy)

        n_train = X_train.shape[0]

        q_values, alpha = extract_alpha_from_cqm_sample(
            best_sample,
            n_train,
            scale
        )

        y_train_pm = (2 * y_train - 1).astype(float)
        balance_value = np.sum(alpha * y_train_pm)

        print("\nq values:")
        print(q_values)

        print("\nAlpha values:")
        print(alpha)

        print("\nSVM balance value sum(alpha_i * y_i):")
        print(balance_value)

        print("\nNumber of nonzero alpha/support vectors:")
        print(np.sum(alpha > 1e-8))

        w = extract_dual_weights_from_alpha(alpha, X_train, y_train)

        b, support_indices = compute_bias_from_support_vectors(
            alpha,
            X_train,
            y_train,
            w,
            C=cfg["C"]
        )

        print("\nBias b:", b)
        print("Support vector indices:", support_indices)

        y_prob = predict(X_test, w, b)
        metrics = compute_metrics(y_test, y_prob)

        runtime = time.time() - start
        completed_jobs += 1
        progress = (completed_jobs / total_models) * 100

        print(
            f"\n[{progress:6.2f}%] "
            f"{name} | "
            f"Acc={metrics['Accuracy']:.4f} | "
            f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
            f"PR-AUC={metrics['PR-AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Precision={metrics['Precision']:.4f} | "
            f"Recall={metrics['Recall']:.4f} | "
            f"Sensitivity={metrics['Sensitivity']:.4f} | "
            f"Specificity={metrics['Specificity']:.4f} | "
            f"Kappa={metrics['Kappa']:.4f} | "
            f"Runtime={runtime:.2f}s | "
            f"CQM_time_limit={used_time_limit}s"
        )

        row = {
            "Model": name,
            "Split": split,
            "Kernel": cfg["kernel"],
            "C": cfg["C"],
            "n_bits": cfg["n_bits"],
            "RBF_Gamma": cfg["rbf_gamma"],
            "Train_Size": X_train.shape[0],
            "Test_Size": X_test.shape[0],
            "Selected_Features": selected_feature_names,
            "Accuracy_mean": metrics["Accuracy"],
            "ROC_AUC_mean": metrics["ROC-AUC"],
            "PR_AUC": metrics["PR-AUC"],
            "F1_mean": metrics["F1"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "Kappa": metrics["Kappa"],
            "Runtime_sec": runtime,
            "CQM_time_limit_sec": used_time_limit,
            "Feasible_Solution": feasible_found,
            "Best_Energy": best_energy,
            "Balance_Value": balance_value,
            "Nonzero_Alpha_Count": int(np.sum(alpha > 1e-8))
        }

        summary.append(row)

        pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

        if VISUALIZE:
            feature_importance_df = run_all_cqm_visualizations(
                sampleset=sampleset,
                K_train=K_train,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                y_prob=y_prob,
                alpha=alpha,
                w=w,
                b=b,
                metrics=metrics,
                selected_feature_names=selected_feature_names,
                scale=scale
            )

            print("\nFeature Importance Table:")
            display(feature_importance_df)


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_df = (
    pd.DataFrame(summary)
    .sort_values("Accuracy_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)

if VISUALIZE:
    plot_runtime_summary(summary_df)

print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>DQM**

In [ ]:
# ============================================================
# Fully D-Wave Hybrid DQM-Based QSVM with Visualizations
# Dataset must already be loaded as: dff
# Target column: LUNG_CANCER
# Solver: D-Wave LeapHybridDQMSampler
# ============================================================

!pip install -q --upgrade-strategy only-if-needed dwave-ocean-sdk matplotlib seaborn networkx

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import time
import os
from getpass import getpass

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score,
    roc_curve, precision_recall_curve
)

import dimod
from dwave.system import LeapHybridDQMSampler


# ============================================================
# D-WAVE TOKEN
# ============================================================

DWAVE_TOKEN = getpass("Paste your D-Wave API token: ")


# ============================================================
# USER CONTROLS
# ============================================================

SPLIT_MODE = "holdout"
N_RUNS = 1
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 100
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "live_dqm_qsvm_results.csv"
n_splits = 1

# DQM-QSVM settings
N_BITS = 1
DEFAULT_C = 0.3
KERNEL_TYPE = "linear"      # "linear" or "rbf"
RBF_GAMMA = 1.0

# DQM cannot add SVM equality constraint directly.
# So we use penalty:
# BALANCE_PENALTY * (sum(alpha_i * y_i))^2
BALANCE_PENALTY = 5.0

# DQM solver time
# None = automatically use minimum time if available
# For stronger final run, try 30, 60, 120
MANUAL_DQM_TIME_LIMIT = None

# Visualization control
VISUALIZE = True


# ============================================================
# LOAD DATA
# dff must exist before running this cell
# ============================================================

if "dff" not in globals():
    raise ValueError("dff not found. Please load your dataset into a DataFrame named dff first.")

if TARGET not in dff.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dff.")

X_df = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()

y_raw = dff[TARGET].values
unique_targets = sorted(pd.Series(y_raw).dropna().unique())

if set(unique_targets) == {0, 1}:
    y = dff[TARGET].astype(int).values
else:
    if len(unique_targets) != 2:
        raise ValueError(f"Target must be binary. Found target values: {unique_targets}")

    target_map = {
        unique_targets[0]: 0,
        unique_targets[1]: 1
    }

    y = pd.Series(y_raw).map(target_map).astype(int).values
    print("\nTarget mapping used:")
    print(target_map)

X_raw = X_df.values.astype(float)

print(dff.info())


# ============================================================
# SELECTED FEATURES
# ============================================================

selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

max_index = max(selected_features_indices)

if max_index >= len(feature_names):
    raise ValueError(
        f"Selected feature index {max_index} is out of range. "
        f"Dataset has only {len(feature_names)} features excluding target."
    )

selected_feature_names = [feature_names[i] for i in selected_features_indices]
X_raw = X_raw[:, selected_features_indices]

print("\nSelected feature names:")
print(selected_feature_names)

print("\nTotal samples:", X_raw.shape[0])
print("Selected feature count:", X_raw.shape[1])


# ============================================================
# BASIC FUNCTIONS
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }


def compute_kernel_matrix(XA, XB=None, kernel="linear", rbf_gamma=1.0):
    if XB is None:
        XB = XA

    if kernel == "linear":
        K = XA @ XB.T

    elif kernel == "rbf":
        XA_sq = np.sum(XA ** 2, axis=1, keepdims=True)
        XB_sq = np.sum(XB ** 2, axis=1, keepdims=True).T
        dists = XA_sq + XB_sq - 2 * (XA @ XB.T)
        K = np.exp(-rbf_gamma * dists)

    else:
        raise ValueError("kernel must be 'linear' or 'rbf'.")

    return K


# ============================================================
# BUILD DQM DUAL QSVM
# ============================================================

def build_dual_qsvm_dqm(
    X,
    y,
    n_bits=3,
    C=0.3,
    balance_penalty=5.0,
    kernel="linear",
    rbf_gamma=1.0
):
    """
    SVM dual objective:

        Maximize:
            sum(alpha_i) - 1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij

        D-Wave minimizes:
            1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij - sum(alpha_i)

    SVM balance condition:

        sum(alpha_i * y_i) = 0

    DQM does not directly support this equality constraint like CQM.
    So we add penalty:

        balance_penalty * (sum(alpha_i * y_i))^2

    DQM alpha representation:

        alpha_i chooses one case from:
        0, scale, 2*scale, ..., q_max*scale

        q_max = 2^n_bits - 1
        scale = C / q_max
    """

    y_pm = (2 * y - 1).astype(float)
    n = X.shape[0]

    K = compute_kernel_matrix(X, kernel=kernel, rbf_gamma=rbf_gamma)

    q_max = (2 ** n_bits) - 1
    num_cases = q_max + 1
    scale = C / q_max

    alpha_values = np.array([case * scale for case in range(num_cases)])

    dqm = dimod.DiscreteQuadraticModel()

    def alpha_var(i):
        return f"alpha_{i}"

    # Add one DQM variable per training sample.
    # Each variable has cases: 0,1,2,...,q_max
    for i in range(n):
        dqm.add_variable(num_cases, label=alpha_var(i))

    # --------------------------------------------------------
    # Linear biases:
    #
    # -alpha_i
    # + 0.5 * alpha_i^2 * y_i^2 * K_ii
    # + balance_penalty * alpha_i^2 * y_i^2
    # --------------------------------------------------------

    for i in range(n):
        linear_biases = []

        for case_i, alpha_i in enumerate(alpha_values):

            svm_linear = -alpha_i

            svm_diagonal = (
                0.5
                * alpha_i
                * alpha_i
                * y_pm[i]
                * y_pm[i]
                * K[i, i]
            )

            penalty_diagonal = (
                balance_penalty
                * alpha_i
                * alpha_i
                * y_pm[i]
                * y_pm[i]
            )

            total_linear = svm_linear + svm_diagonal + penalty_diagonal

            linear_biases.append(float(total_linear))

        dqm.set_linear(alpha_var(i), linear_biases)

    # --------------------------------------------------------
    # Quadratic biases for i < j:
    #
    # SVM cross term:
    # alpha_i * alpha_j * y_i * y_j * K_ij
    #
    # Penalty cross term:
    # 2 * balance_penalty * alpha_i * alpha_j * y_i * y_j
    # --------------------------------------------------------

    for i in range(n):
        for j in range(i + 1, n):

            quadratic_biases = {}

            for case_i, alpha_i in enumerate(alpha_values):
                for case_j, alpha_j in enumerate(alpha_values):

                    svm_cross = (
                        alpha_i
                        * alpha_j
                        * y_pm[i]
                        * y_pm[j]
                        * K[i, j]
                    )

                    penalty_cross = (
                        2.0
                        * balance_penalty
                        * alpha_i
                        * alpha_j
                        * y_pm[i]
                        * y_pm[j]
                    )

                    total_quadratic = svm_cross + penalty_cross

                    quadratic_biases[(case_i, case_j)] = float(total_quadratic)

            dqm.set_quadratic(alpha_var(i), alpha_var(j), quadratic_biases)

    return dqm, K, scale, q_max, alpha_values


# ============================================================
# SOLVE DQM
# ============================================================

def solve_dqm_qsvm(
    dqm,
    token=None,
    manual_time_limit=None,
    label="DQM-QSVM"
):
    if token is None or token.strip() == "":
        sampler = LeapHybridDQMSampler()
    else:
        sampler = LeapHybridDQMSampler(token=token)

    try:
        min_time = sampler.min_time_limit(dqm)
    except Exception:
        min_time = 5

    if manual_time_limit is None:
        time_limit = max(5, int(np.ceil(min_time)))
    else:
        time_limit = max(5, int(np.ceil(min_time)), int(manual_time_limit))

    print("\nDQM solver:", sampler.solver.name)
    print("Minimum required time:", min_time)
    print("Using time_limit:", time_limit)

    sampleset = sampler.sample_dqm(
        dqm,
        time_limit=time_limit,
        label=label
    )

    best_sample = sampleset.first.sample
    best_energy = sampleset.first.energy

    return best_sample, best_energy, sampleset, time_limit, sampler.solver.name


# ============================================================
# EXTRACT ALPHA, OBJECTIVE, WEIGHT, BIAS
# ============================================================

def extract_alpha_from_dqm_sample(sample, n, scale):
    q_values = np.zeros(n)

    for i in range(n):
        q_values[i] = int(sample[f"alpha_{i}"])

    alpha = q_values * scale

    return q_values, alpha


def true_svm_dual_objective(alpha, y_train, K_train):
    y_pm = (2 * y_train - 1).astype(float)

    quad = 0.0

    for i in range(len(alpha)):
        for j in range(len(alpha)):
            quad += (
                0.5
                * alpha[i]
                * alpha[j]
                * y_pm[i]
                * y_pm[j]
                * K_train[i, j]
            )

    linear = -np.sum(alpha)

    return quad + linear


def select_best_dqm_solution(
    sampleset,
    n_train,
    scale,
    y_train,
    K_train,
    balance_tolerance=1e-9
):
    y_pm = (2 * y_train - 1).astype(float)

    candidates = []

    for row in sampleset.data(["sample", "energy", "num_occurrences"]):
        sample = row.sample
        energy = row.energy
        num_occurrences = row.num_occurrences

        q_values, alpha = extract_alpha_from_dqm_sample(
            sample,
            n=n_train,
            scale=scale
        )

        balance_value = float(np.sum(alpha * y_pm))
        balance_abs = abs(balance_value)
        true_obj = true_svm_dual_objective(alpha, y_train, K_train)
        nonzero_alpha_count = int(np.sum(alpha > 1e-8))

        candidates.append({
            "sample": sample,
            "energy": energy,
            "num_occurrences": num_occurrences,
            "q_values": q_values,
            "alpha": alpha,
            "balance_value": balance_value,
            "balance_abs": balance_abs,
            "true_svm_objective": true_obj,
            "nonzero_alpha_count": nonzero_alpha_count
        })

    exact_balance = [
        c for c in candidates
        if c["balance_abs"] <= balance_tolerance and c["nonzero_alpha_count"] > 0
    ]

    if len(exact_balance) > 0:
        best = min(exact_balance, key=lambda c: c["true_svm_objective"])
        selection_note = "Exact-balanced DQM solution selected."
    else:
        best = min(candidates, key=lambda c: (c["balance_abs"], c["energy"]))
        selection_note = "No exact-balanced solution found. Selected smallest balance violation."

    return best, candidates, selection_note


def compute_bias_dual(alpha, y_train, K_train, C=0.3):
    y_pm = (2 * y_train - 1).astype(float)

    support_indices = np.where(alpha > 1e-8)[0]
    margin_indices = np.where((alpha > 1e-8) & (alpha < C - 1e-8))[0]

    if len(margin_indices) > 0:
        selected = margin_indices
    else:
        selected = support_indices

    if len(selected) == 0:
        return 0.0, support_indices

    b_values = []

    for i in selected:
        decision_without_b = np.sum(alpha * y_pm * K_train[i, :])
        b_i = y_pm[i] - decision_without_b
        b_values.append(b_i)

    b = float(np.mean(b_values))

    return b, support_indices


def decision_function_dual(
    X_query,
    X_train,
    alpha,
    y_train,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    y_pm = (2 * y_train - 1).astype(float)

    K_query = compute_kernel_matrix(
        X_query,
        X_train,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    scores = K_query @ (alpha * y_pm) + b

    return scores


def predict_probability_dual(
    X_query,
    X_train,
    alpha,
    y_train,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    scores = decision_function_dual(
        X_query,
        X_train,
        alpha,
        y_train,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    return sigmoid(scores)


def extract_linear_weight_from_alpha(alpha, X_train, y_train):
    y_pm = (2 * y_train - 1).astype(float)
    w = (alpha * y_pm) @ X_train
    return w


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_dqm_energy_convergence(sampleset):
    energies = np.asarray(sampleset.record.energy)
    best = np.minimum.accumulate(energies)

    plt.figure(figsize=(7, 4))
    plt.plot(best, marker="o", linewidth=1)
    plt.title("DQM Energy Convergence")
    plt.xlabel("Returned Sample Index")
    plt.ylabel("Best Energy So Far")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_dqm_energy_hist(sampleset):
    energies = np.asarray(sampleset.record.energy)

    plt.figure(figsize=(7, 4))
    plt.hist(energies, bins=30)
    plt.title("DQM Energy Distribution")
    plt.xlabel("Energy")
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_dqm_solution_frequency(sampleset, top_k=10):
    try:
        df_samples = sampleset.to_pandas_dataframe()
    except Exception as e:
        print("Cannot convert sampleset to dataframe:", e)
        return

    var_cols = [
        col for col in df_samples.columns
        if str(col).startswith("alpha_")
    ]

    if len(var_cols) == 0:
        print("No alpha variables found for DQM solution frequency plot.")
        return

    if "num_occurrences" not in df_samples.columns:
        df_samples["num_occurrences"] = 1

    df_samples["_solution_signature"] = (
        df_samples[var_cols]
        .astype(int)
        .astype(str)
        .agg(",".join, axis=1)
    )

    counts = (
        df_samples
        .groupby("_solution_signature")["num_occurrences"]
        .sum()
        .sort_values(ascending=False)
        .head(top_k)
    )

    short_labels = [
        s[:80] + "..." if len(s) > 80 else s
        for s in counts.index
    ]

    plt.figure(figsize=(10, 5))
    plt.bar(range(len(counts)), counts.values)
    plt.xticks(range(len(counts)), short_labels, rotation=45, ha="right")
    plt.title(f"Top {top_k} DQM Solution Frequencies")
    plt.ylabel("Occurrences")
    plt.tight_layout()
    plt.show()


def plot_kernel_matrix_heatmap(K, title="Training Kernel Matrix"):
    plt.figure(figsize=(7, 6))
    sns.heatmap(K, cmap="viridis")
    plt.title(title)
    plt.xlabel("Training Sample Index")
    plt.ylabel("Training Sample Index")
    plt.show()


def plot_dual_objective_matrix(K, y_train, scale):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    plt.figure(figsize=(7, 6))
    sns.heatmap(Q, cmap="coolwarm", center=0)
    plt.title("DQM Dual QSVM Interaction Matrix")
    plt.xlabel("Training Sample j")
    plt.ylabel("Training Sample i")
    plt.show()


def plot_dual_interaction_graph(K, y_train, scale, max_edges=80):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    n = Q.shape[0]
    G = nx.Graph()

    for i in range(n):
        G.add_node(f"alpha_{i}")

    edges = []

    for i in range(n):
        for j in range(i + 1, n):
            weight = abs(Q[i, j])
            if weight > 0:
                edges.append((i, j, weight))

    edges = sorted(edges, key=lambda x: x[2], reverse=True)[:max_edges]

    for i, j, weight in edges:
        G.add_edge(f"alpha_{i}", f"alpha_{j}", weight=weight)

    plt.figure(figsize=(9, 7))
    pos = nx.spring_layout(G, seed=42)

    nx.draw(
        G,
        pos,
        with_labels=True,
        node_size=450,
        font_size=8,
        edge_color="gray"
    )

    plt.title(f"DQM Interaction Graph: Top {max_edges} Edges")
    plt.show()


def plot_alpha_values(alpha):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(alpha)), alpha)
    plt.title("Optimized Alpha Values")
    plt.xlabel("Training Sample Index")
    plt.ylabel("Alpha")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_alpha_cases(q_values):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(q_values)), q_values)
    plt.title("Selected DQM Alpha Cases")
    plt.xlabel("Training Sample Index")
    plt.ylabel("Selected Case q_i")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_alpha_by_class(alpha, y_train):
    df_alpha = pd.DataFrame({
        "Alpha": alpha,
        "Class": y_train.astype(int)
    })

    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_alpha, x="Class", y="Alpha")
    sns.stripplot(data=df_alpha, x="Class", y="Alpha", alpha=0.7)
    plt.title("Alpha Distribution by Class")
    plt.xlabel("Class")
    plt.ylabel("Alpha")
    plt.show()


def plot_support_vectors_pca(X_train, y_train, alpha, title="Support Vectors in PCA Space"):
    support_idx = alpha > 1e-8

    if X_train.shape[1] > 2:
        X_vis = PCA(n_components=2).fit_transform(X_train)
    else:
        X_vis = X_train

    plt.figure(figsize=(7, 5))
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        s=50,
        label="Training Samples"
    )

    plt.scatter(
        X_vis[support_idx, 0],
        X_vis[support_idx, 1],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Support Vectors"
    )

    plt.title(title)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_weight_importance(w, selected_feature_names):
    importance = np.abs(w)

    df_imp = pd.DataFrame({
        "Feature": selected_feature_names,
        "Weight": w,
        "Abs_Weight": importance
    }).sort_values("Abs_Weight", ascending=False)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=df_imp, x="Abs_Weight", y="Feature")
    plt.title("Feature Importance from DQM-QSVM Linear Weight Vector")
    plt.xlabel("|Weight|")
    plt.ylabel("Feature")
    plt.show()

    return df_imp


def plot_confusion_matrix_custom(y_true, y_prob, threshold=0.5, title="Confusion Matrix"):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def plot_roc_curve_custom(y_true, y_prob):
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_score = roc_auc_score(y_true, y_prob)
    except Exception as e:
        print("ROC curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_pr_curve_custom(y_true, y_prob):
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap_score = average_precision_score(y_true, y_prob)
    except Exception as e:
        print("PR curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC/AP = {ap_score:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_prediction_distribution(y_true, y_prob):
    df_pred = pd.DataFrame({
        "Probability": y_prob,
        "Class": y_true.astype(int)
    })

    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_pred, x="Probability", hue="Class", bins=30, kde=True)
    plt.title("Predicted Probability Distribution")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.show()


def plot_metrics_bar(metrics):
    selected_metrics = {
        "Accuracy": metrics.get("Accuracy", np.nan),
        "Precision": metrics.get("Precision", np.nan),
        "Recall": metrics.get("Recall", np.nan),
        "F1": metrics.get("F1", np.nan),
        "ROC-AUC": metrics.get("ROC-AUC", np.nan),
        "PR-AUC": metrics.get("PR-AUC", np.nan),
        "Sensitivity": metrics.get("Sensitivity", np.nan),
        "Specificity": metrics.get("Specificity", np.nan),
        "Kappa": metrics.get("Kappa", np.nan)
    }

    names = list(selected_metrics.keys())
    values = list(selected_metrics.values())

    plt.figure(figsize=(10, 4))
    plt.bar(names, values)
    plt.ylim(0, 1.05)
    plt.title("Model Performance Metrics")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_decision_boundary_pca_dual(
    X_train,
    y_train,
    alpha,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    if X_train.shape[1] > 2:
        pca = PCA(n_components=2)
        X_vis = pca.fit_transform(X_train)
    else:
        pca = None
        X_vis = X_train

    x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
    y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid_2d = np.c_[xx.ravel(), yy.ravel()]

    if pca is not None:
        grid_full = pca.inverse_transform(grid_2d)
    else:
        grid_full = grid_2d

    scores = decision_function_dual(
        grid_full,
        X_train,
        alpha,
        y_train,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    z = sigmoid(scores)
    z = z.reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, z, levels=50, alpha=0.8)
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        edgecolors="black",
        s=45
    )
    plt.title("Decision Boundary in PCA Space")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(label="Predicted Probability")
    plt.show()


def plot_runtime_summary(summary_df):
    if "Runtime_sec" not in summary_df.columns:
        print("Runtime_sec column not found.")
        return

    plt.figure(figsize=(8, 4))
    sns.barplot(data=summary_df, x="Model", y="Runtime_sec")
    plt.title("Runtime Comparison")
    plt.xlabel("Model")
    plt.ylabel("Runtime Second")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def run_all_dqm_visualizations(
    sampleset,
    K_train,
    X_train,
    y_train,
    X_test,
    y_test,
    y_prob,
    q_values,
    alpha,
    w,
    b,
    metrics,
    selected_feature_names,
    scale,
    kernel,
    rbf_gamma
):
    print("\n========== DQM / Solver Visualizations ==========")
    plot_dqm_energy_convergence(sampleset)
    plot_dqm_energy_hist(sampleset)
    plot_dqm_solution_frequency(sampleset, top_k=10)

    print("\n========== Kernel / Objective Visualizations ==========")
    plot_kernel_matrix_heatmap(K_train, title="Training Kernel Matrix")
    plot_dual_objective_matrix(K_train, y_train, scale)
    plot_dual_interaction_graph(K_train, y_train, scale, max_edges=80)

    print("\n========== Alpha / Support Vector Visualizations ==========")
    plot_alpha_cases(q_values)
    plot_alpha_values(alpha)
    plot_alpha_by_class(alpha, y_train)
    plot_support_vectors_pca(X_train, y_train, alpha)

    print("\n========== Model Performance Visualizations ==========")
    plot_confusion_matrix_custom(y_test, y_prob, threshold=0.5)
    plot_roc_curve_custom(y_test, y_prob)
    plot_pr_curve_custom(y_test, y_prob)
    plot_prediction_distribution(y_test, y_prob)
    plot_metrics_bar(metrics)

    print("\n========== Feature / Decision Boundary Visualizations ==========")

    feature_importance_df = None

    if kernel == "linear" and w is not None:
        feature_importance_df = plot_weight_importance(w, selected_feature_names)
    else:
        print("Feature weight importance is only directly available for linear kernel.")

    plot_decision_boundary_pca_dual(
        X_train,
        y_train,
        alpha,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    return feature_importance_df


# ============================================================
# MAIN LOOP
# ============================================================

summary = []

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

dual_configs = {
    "DW_DQM_QSVM_SoftMargin_LowC": dict(
        kernel=KERNEL_TYPE,
        C=DEFAULT_C,
        n_bits=N_BITS,
        rbf_gamma=RBF_GAMMA,
        balance_penalty=BALANCE_PENALTY
    )
}

total_models = len(dual_configs) * n_splits
completed_jobs = 0

for split in range(1, n_splits + 1):

    print("\n============================================================")
    print(f"Split {split}/{n_splits}")
    print("============================================================")

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * 42
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    print("Train shape:", X_train.shape)
    print("Test shape :", X_test.shape)

    for name, cfg in dual_configs.items():

        start = time.time()

        print("\n------------------------------------------------------------")
        print("Running model:", name)
        print("Config:", cfg)
        print("------------------------------------------------------------")

        dqm, K_train, scale, q_max, alpha_values = build_dual_qsvm_dqm(
            X_train,
            y_train,
            n_bits=cfg["n_bits"],
            C=cfg["C"],
            balance_penalty=cfg["balance_penalty"],
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"]
        )

        print("\nDQM created.")
        print("Number of DQM variables:", X_train.shape[0])
        print("Cases per variable:", q_max + 1)
        print("q_max:", q_max)
        print("alpha scale:", scale)
        print("alpha values:", alpha_values)
        print("balance penalty:", cfg["balance_penalty"])

        raw_best_sample, raw_best_energy, sampleset, used_time_limit, solver_name = solve_dqm_qsvm(
            dqm,
            token=DWAVE_TOKEN,
            manual_time_limit=MANUAL_DQM_TIME_LIMIT,
            label=f"{name}_split_{split}"
        )

        print("\nRaw best DQM energy:", raw_best_energy)

        n_train = X_train.shape[0]

        best_candidate, candidates, selection_note = select_best_dqm_solution(
            sampleset=sampleset,
            n_train=n_train,
            scale=scale,
            y_train=y_train,
            K_train=K_train,
            balance_tolerance=1e-9
        )

        print("\nSelection note:")
        print(selection_note)

        q_values = best_candidate["q_values"]
        alpha = best_candidate["alpha"]
        balance_value = best_candidate["balance_value"]
        best_energy = best_candidate["energy"]
        true_obj = best_candidate["true_svm_objective"]

        print("\nq values / selected DQM cases:")
        print(q_values)

        print("\nAlpha values:")
        print(alpha)

        print("\nSVM balance value sum(alpha_i * y_i):")
        print(balance_value)

        print("\nTrue SVM dual objective without penalty:")
        print(true_obj)

        print("\nSelected DQM energy:")
        print(best_energy)

        print("\nNumber of nonzero alpha/support vectors:")
        print(np.sum(alpha > 1e-8))

        b, support_indices = compute_bias_dual(
            alpha,
            y_train,
            K_train,
            C=cfg["C"]
        )

        print("\nBias b:", b)
        print("Support vector indices:", support_indices)

        if cfg["kernel"] == "linear":
            w = extract_linear_weight_from_alpha(alpha, X_train, y_train)
        else:
            w = None

        y_prob = predict_probability_dual(
            X_test,
            X_train,
            alpha,
            y_train,
            b,
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"]
        )

        metrics = compute_metrics(y_test, y_prob)

        runtime = time.time() - start
        completed_jobs += 1
        progress = (completed_jobs / total_models) * 100

        print(
            f"\n[{progress:6.2f}%] "
            f"{name} | "
            f"Acc={metrics['Accuracy']:.4f} | "
            f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
            f"PR-AUC={metrics['PR-AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Precision={metrics['Precision']:.4f} | "
            f"Recall={metrics['Recall']:.4f} | "
            f"Sensitivity={metrics['Sensitivity']:.4f} | "
            f"Specificity={metrics['Specificity']:.4f} | "
            f"Kappa={metrics['Kappa']:.4f} | "
            f"Runtime={runtime:.2f}s | "
            f"Solver={solver_name} | "
            f"DQM_time_limit={used_time_limit}"
        )

        row = {
            "Model": name,
            "Split": split,
            "Solver_Name": solver_name,
            "Kernel": cfg["kernel"],
            "C": cfg["C"],
            "n_bits": cfg["n_bits"],
            "RBF_Gamma": cfg["rbf_gamma"],
            "Balance_Penalty": cfg["balance_penalty"],
            "Train_Size": X_train.shape[0],
            "Test_Size": X_test.shape[0],
            "Selected_Features": selected_feature_names,
            "DQM_Variables": X_train.shape[0],
            "DQM_Cases_Per_Variable": q_max + 1,
            "Accuracy_mean": metrics["Accuracy"],
            "ROC_AUC_mean": metrics["ROC-AUC"],
            "PR_AUC": metrics["PR-AUC"],
            "F1_mean": metrics["F1"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "Kappa": metrics["Kappa"],
            "Runtime_sec": runtime,
            "DQM_time_limit_sec": used_time_limit,
            "Raw_Best_Energy": raw_best_energy,
            "Selected_Best_Energy": best_energy,
            "True_SVM_Objective": true_obj,
            "Balance_Value": balance_value,
            "Balance_Abs": abs(balance_value),
            "Nonzero_Alpha_Count": int(np.sum(alpha > 1e-8)),
            "Selection_Note": selection_note
        }

        summary.append(row)

        pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

        if VISUALIZE:
            feature_importance_df = run_all_dqm_visualizations(
                sampleset=sampleset,
                K_train=K_train,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                y_prob=y_prob,
                q_values=q_values,
                alpha=alpha,
                w=w,
                b=b,
                metrics=metrics,
                selected_feature_names=selected_feature_names,
                scale=scale,
                kernel=cfg["kernel"],
                rbf_gamma=cfg["rbf_gamma"]
            )

            if feature_importance_df is not None:
                print("\nFeature Importance Table:")
                display(feature_importance_df)


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_df = (
    pd.DataFrame(summary)
    .sort_values("Accuracy_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)

if VISUALIZE:
    plot_runtime_summary(summary_df)

print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>nonlinear**

In [ ]:
# ============================================================
# Fully D-Wave Hybrid NL-Based QSVM with Visualizations
# Dataset must already be loaded as: dff
# Target column: LUNG_CANCER
# Solver: D-Wave LeapHybridNLSampler / Stride NL Solver
# ============================================================

!pip install -q --upgrade-strategy only-if-needed dwave-ocean-sdk dwave-optimization matplotlib seaborn networkx

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import time
import os
from getpass import getpass

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score,
    roc_curve, precision_recall_curve
)

try:
    from dwave.optimization import Model
except Exception:
    from dwave.optimization.model import Model

from dwave.system import LeapHybridNLSampler


# ============================================================
# D-WAVE TOKEN
# ============================================================

DWAVE_TOKEN = getpass("Paste your D-Wave API token: ")


# ============================================================
# USER CONTROLS
# ============================================================

SPLIT_MODE = "holdout"
N_RUNS = 1
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 100
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "live_nl_qsvm_results.csv"
n_splits = 1

# NL-QSVM settings
N_BITS = 1
DEFAULT_C = 0.3
KERNEL_TYPE = "linear"      # "linear" or "rbf"
RBF_GAMMA = 1.0

# None = use D-Wave estimated minimum time.
# For stronger final run, try 30, 60, 120.
MANUAL_NL_TIME_LIMIT = None

# Prevent useless all-zero alpha solution
USE_NONZERO_ALPHA_CONSTRAINT = True
MIN_TOTAL_Q = 2

# Visualization control
VISUALIZE = True


# ============================================================
# LOAD DATA
# dff must exist before running this cell
# ============================================================

if "dff" not in globals():
    raise ValueError("dff not found. Please load your dataset into a DataFrame named dff first.")

if TARGET not in dff.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dff.")

X_df = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()

y_raw = dff[TARGET].values
unique_targets = sorted(pd.Series(y_raw).dropna().unique())

if set(unique_targets) == {0, 1}:
    y = dff[TARGET].astype(int).values
else:
    if len(unique_targets) != 2:
        raise ValueError(f"Target must be binary. Found target values: {unique_targets}")

    target_map = {
        unique_targets[0]: 0,
        unique_targets[1]: 1
    }

    y = pd.Series(y_raw).map(target_map).astype(int).values
    print("\nTarget mapping used:")
    print(target_map)

X_raw = X_df.values.astype(float)

print(dff.info())


# ============================================================
# SELECTED FEATURES
# ============================================================

selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

max_index = max(selected_features_indices)

if max_index >= len(feature_names):
    raise ValueError(
        f"Selected feature index {max_index} is out of range. "
        f"Dataset has only {len(feature_names)} features excluding target."
    )

selected_feature_names = [feature_names[i] for i in selected_features_indices]
X_raw = X_raw[:, selected_features_indices]

print("\nSelected feature names:")
print(selected_feature_names)

print("\nTotal samples:", X_raw.shape[0])
print("Selected feature count:", X_raw.shape[1])


# ============================================================
# BASIC FUNCTIONS
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }


def compute_kernel_matrix(XA, XB=None, kernel="linear", rbf_gamma=1.0):
    if XB is None:
        XB = XA

    if kernel == "linear":
        K = XA @ XB.T

    elif kernel == "rbf":
        XA_sq = np.sum(XA ** 2, axis=1, keepdims=True)
        XB_sq = np.sum(XB ** 2, axis=1, keepdims=True).T
        dists = XA_sq + XB_sq - 2 * (XA @ XB.T)
        K = np.exp(-rbf_gamma * dists)

    else:
        raise ValueError("kernel must be 'linear' or 'rbf'.")

    return K


# ============================================================
# BUILD NL DUAL QSVM
# ============================================================

def build_dual_qsvm_nl(
    X,
    y,
    n_bits=3,
    C=0.3,
    kernel="linear",
    rbf_gamma=1.0,
    use_nonzero_alpha_constraint=True,
    min_total_q=2
):
    """
    SVM dual objective:

        Maximize:
            sum(alpha_i) - 1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij

        D-Wave minimizes:
            1/2 * sum_i sum_j alpha_i alpha_j y_i y_j K_ij - sum(alpha_i)

    Constraint:
        sum(alpha_i * y_i) = 0

    NL discretization:
        alpha_i = q_i * scale
        q_i ∈ {0, 1, ..., 2^n_bits - 1}
        scale = C / (2^n_bits - 1)
    """

    y_pm = (2 * y - 1).astype(int)
    n = X.shape[0]

    K = compute_kernel_matrix(X, kernel=kernel, rbf_gamma=rbf_gamma)

    q_max = (2 ** n_bits) - 1
    scale = C / q_max

    model = Model()

    # Integer decision variables q_i
    try:
        q = model.integer(n, lower_bound=0, upper_bound=q_max)
    except TypeError:
        q = model.integer((n,), lower_bound=0, upper_bound=q_max)

    objective = 0

    # Quadratic SVM objective:
    # Use upper-triangular form to avoid double-counting.
    for i in range(n):
        for j in range(i, n):

            if i == j:
                coef = 0.5 * y_pm[i] * y_pm[j] * K[i, j] * (scale ** 2)
            else:
                coef = y_pm[i] * y_pm[j] * K[i, j] * (scale ** 2)

            objective = objective + float(coef) * q[i] * q[j]

    # Linear term: -sum(alpha_i)
    for i in range(n):
        objective = objective + float(-scale) * q[i]

    model.minimize(objective)

    # SVM equality constraint:
    # sum(alpha_i * y_i) = 0
    # Since alpha_i = q_i * scale:
    # sum(q_i * y_i) = 0
    balance_constraint = 0

    for i in range(n):
        balance_constraint = balance_constraint + int(y_pm[i]) * q[i]

    model.add_constraint(balance_constraint == 0)

    # Prevent all-zero alpha solution
    if use_nonzero_alpha_constraint:
        total_q = 0

        for i in range(n):
            total_q = total_q + q[i]

        model.add_constraint(total_q >= min_total_q)

    try:
        model.lock()
    except Exception:
        pass

    return model, q, K, scale, q_max


# ============================================================
# SOLVE NL MODEL
# ============================================================

def solve_nl_qsvm(model, token=None, manual_time_limit=None, label="NL-QSVM"):
    if token is None or token.strip() == "":
        sampler = LeapHybridNLSampler()
    else:
        sampler = LeapHybridNLSampler(token=token)

    try:
        min_time = sampler.estimated_min_time_limit(model)
    except Exception:
        min_time = 5

    if manual_time_limit is None:
        time_limit = max(5, int(np.ceil(min_time)))
    else:
        time_limit = max(5, int(np.ceil(min_time)), int(manual_time_limit))

    print("\nNL solver:", sampler.solver.name)
    print("Estimated minimum required time:", min_time)
    print("Using time_limit:", time_limit)

    try:
        future = sampler.sample(
            model,
            time_limit=time_limit,
            label=label
        )
    except TypeError:
        future = sampler.sample(
            model,
            label=label
        )

    if hasattr(future, "result"):
        result = future.result()
    else:
        result = future

    return result, time_limit, sampler.solver.name


# ============================================================
# NL STATE EXTRACTION
# ============================================================

def safe_float(value):
    try:
        return float(np.asarray(value).squeeze())
    except Exception:
        return float("inf")


def get_num_model_states(model):
    try:
        return model.states.size()
    except Exception:
        return 1


def check_feasible_state(model, state_index):
    try:
        constraints = list(model.iter_constraints())

        if len(constraints) == 0:
            return True

        for constraint in constraints:
            value = constraint.state(state_index)

            if not bool(np.all(value)):
                return False

        return True

    except Exception:
        return True


def get_objective_value(model, state_index):
    try:
        return safe_float(model.objective.state(state_index))
    except Exception:
        return float("inf")


def select_best_nl_state(model):
    num_states = get_num_model_states(model)

    print("\nNumber of returned NL states:", num_states)

    state_rows = []
    best_state = None
    best_objective = float("inf")

    for state_index in range(num_states):
        feasible = check_feasible_state(model, state_index)
        obj = get_objective_value(model, state_index)

        state_rows.append({
            "state": state_index,
            "feasible": feasible,
            "objective": obj
        })

        print(f"State {state_index}: feasible={feasible}, objective={obj}")

        if feasible and obj < best_objective:
            best_objective = obj
            best_state = state_index

    if best_state is None:
        print("\nNo feasible state found. Selecting lowest-objective returned state.")

        best_state = 0
        best_objective = float("inf")

        for row in state_rows:
            if row["objective"] < best_objective:
                best_objective = row["objective"]
                best_state = row["state"]

    print("\nSelected best NL state:", best_state)
    print("Best objective:", best_objective)

    return best_state, best_objective, pd.DataFrame(state_rows)


def extract_alpha_from_nl_state(q, state_index, scale):
    try:
        q_solution = np.asarray(q.state(state_index), dtype=float)
    except Exception:
        q_solution = np.asarray(q.state(), dtype=float)

    q_solution = np.rint(q_solution).astype(int)
    alpha = q_solution * scale

    return q_solution, alpha


# ============================================================
# QSVM DECISION FUNCTIONS
# ============================================================

def true_svm_dual_objective(alpha, y_train, K_train):
    y_pm = (2 * y_train - 1).astype(float)

    quad = 0.0

    for i in range(len(alpha)):
        for j in range(len(alpha)):
            quad += (
                0.5
                * alpha[i]
                * alpha[j]
                * y_pm[i]
                * y_pm[j]
                * K_train[i, j]
            )

    linear = -np.sum(alpha)

    return quad + linear


def compute_bias_dual(alpha, y_train, K_train, C=0.3):
    y_pm = (2 * y_train - 1).astype(float)

    support_indices = np.where(alpha > 1e-8)[0]
    margin_indices = np.where((alpha > 1e-8) & (alpha < C - 1e-8))[0]

    if len(margin_indices) > 0:
        selected = margin_indices
    else:
        selected = support_indices

    if len(selected) == 0:
        return 0.0, support_indices

    b_values = []

    for i in selected:
        decision_without_b = np.sum(alpha * y_pm * K_train[i, :])
        b_i = y_pm[i] - decision_without_b
        b_values.append(b_i)

    b = float(np.mean(b_values))

    return b, support_indices


def decision_function_dual(
    X_query,
    X_train,
    alpha,
    y_train,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    y_pm = (2 * y_train - 1).astype(float)

    K_query = compute_kernel_matrix(
        X_query,
        X_train,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    scores = K_query @ (alpha * y_pm) + b

    return scores


def predict_probability_dual(
    X_query,
    X_train,
    alpha,
    y_train,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    scores = decision_function_dual(
        X_query,
        X_train,
        alpha,
        y_train,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    return sigmoid(scores)


def extract_linear_weight_from_alpha(alpha, X_train, y_train):
    y_pm = (2 * y_train - 1).astype(float)
    w = (alpha * y_pm) @ X_train
    return w


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_nl_objective_convergence(state_df):
    if state_df is None or len(state_df) == 0:
        print("No NL state data available.")
        return

    objectives = state_df["objective"].values
    best = np.minimum.accumulate(objectives)

    plt.figure(figsize=(7, 4))
    plt.plot(best, marker="o", linewidth=1)
    plt.title("NL Objective Convergence")
    plt.xlabel("Returned State Index")
    plt.ylabel("Best Objective So Far")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_nl_objective_hist(state_df):
    if state_df is None or len(state_df) == 0:
        print("No NL state data available.")
        return

    plt.figure(figsize=(7, 4))
    plt.hist(state_df["objective"].values, bins=30)
    plt.title("NL Objective Distribution")
    plt.xlabel("Objective Value")
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_nl_feasibility(state_df):
    if state_df is None or len(state_df) == 0:
        print("No NL state data available.")
        return

    counts = state_df["feasible"].value_counts()

    labels = ["Infeasible", "Feasible"]
    values = [
        counts.get(False, 0),
        counts.get(True, 0)
    ]

    plt.figure(figsize=(5, 4))
    plt.bar(labels, values)
    plt.title("NL Feasible vs Infeasible States")
    plt.ylabel("Number of States")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_kernel_matrix_heatmap(K, title="Training Kernel Matrix"):
    plt.figure(figsize=(7, 6))
    sns.heatmap(K, cmap="viridis")
    plt.title(title)
    plt.xlabel("Training Sample Index")
    plt.ylabel("Training Sample Index")
    plt.show()


def plot_dual_objective_matrix(K, y_train, scale):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    plt.figure(figsize=(7, 6))
    sns.heatmap(Q, cmap="coolwarm", center=0)
    plt.title("NL Dual QSVM Interaction Matrix")
    plt.xlabel("Training Sample j")
    plt.ylabel("Training Sample i")
    plt.show()


def plot_dual_interaction_graph(K, y_train, scale, max_edges=80):
    y_pm = (2 * y_train - 1).astype(float)
    Q = np.outer(y_pm, y_pm) * K * (scale ** 2)

    n = Q.shape[0]
    G = nx.Graph()

    for i in range(n):
        G.add_node(f"q_{i}")

    edges = []

    for i in range(n):
        for j in range(i + 1, n):
            weight = abs(Q[i, j])
            if weight > 0:
                edges.append((i, j, weight))

    edges = sorted(edges, key=lambda x: x[2], reverse=True)[:max_edges]

    for i, j, weight in edges:
        G.add_edge(f"q_{i}", f"q_{j}", weight=weight)

    plt.figure(figsize=(9, 7))
    pos = nx.spring_layout(G, seed=42)

    nx.draw(
        G,
        pos,
        with_labels=True,
        node_size=450,
        font_size=8,
        edge_color="gray"
    )

    plt.title(f"NL Dual Interaction Graph: Top {max_edges} Edges")
    plt.show()


def plot_alpha_values(alpha):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(alpha)), alpha)
    plt.title("Optimized Alpha Values")
    plt.xlabel("Training Sample Index")
    plt.ylabel("Alpha")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_q_values(q_values):
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(q_values)), q_values)
    plt.title("Optimized Integer q Values")
    plt.xlabel("Training Sample Index")
    plt.ylabel("q_i")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def plot_alpha_by_class(alpha, y_train):
    df_alpha = pd.DataFrame({
        "Alpha": alpha,
        "Class": y_train.astype(int)
    })

    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_alpha, x="Class", y="Alpha")
    sns.stripplot(data=df_alpha, x="Class", y="Alpha", alpha=0.7)
    plt.title("Alpha Distribution by Class")
    plt.xlabel("Class")
    plt.ylabel("Alpha")
    plt.show()


def plot_support_vectors_pca(X_train, y_train, alpha, title="Support Vectors in PCA Space"):
    support_idx = alpha > 1e-8

    if X_train.shape[1] > 2:
        X_vis = PCA(n_components=2).fit_transform(X_train)
    else:
        X_vis = X_train

    plt.figure(figsize=(7, 5))
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        s=50,
        label="Training Samples"
    )

    plt.scatter(
        X_vis[support_idx, 0],
        X_vis[support_idx, 1],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Support Vectors"
    )

    plt.title(title)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_weight_importance(w, selected_feature_names):
    importance = np.abs(w)

    df_imp = pd.DataFrame({
        "Feature": selected_feature_names,
        "Weight": w,
        "Abs_Weight": importance
    }).sort_values("Abs_Weight", ascending=False)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=df_imp, x="Abs_Weight", y="Feature")
    plt.title("Feature Importance from NL-QSVM Linear Weight Vector")
    plt.xlabel("|Weight|")
    plt.ylabel("Feature")
    plt.show()

    return df_imp


def plot_confusion_matrix_custom(y_true, y_prob, threshold=0.5, title="Confusion Matrix"):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def plot_roc_curve_custom(y_true, y_prob):
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_score = roc_auc_score(y_true, y_prob)
    except Exception as e:
        print("ROC curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_pr_curve_custom(y_true, y_prob):
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap_score = average_precision_score(y_true, y_prob)
    except Exception as e:
        print("PR curve cannot be plotted:", e)
        return

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC/AP = {ap_score:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_prediction_distribution(y_true, y_prob):
    df_pred = pd.DataFrame({
        "Probability": y_prob,
        "Class": y_true.astype(int)
    })

    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_pred, x="Probability", hue="Class", bins=30, kde=True)
    plt.title("Predicted Probability Distribution")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.show()


def plot_metrics_bar(metrics):
    selected_metrics = {
        "Accuracy": metrics.get("Accuracy", np.nan),
        "Precision": metrics.get("Precision", np.nan),
        "Recall": metrics.get("Recall", np.nan),
        "F1": metrics.get("F1", np.nan),
        "ROC-AUC": metrics.get("ROC-AUC", np.nan),
        "PR-AUC": metrics.get("PR-AUC", np.nan),
        "Sensitivity": metrics.get("Sensitivity", np.nan),
        "Specificity": metrics.get("Specificity", np.nan),
        "Kappa": metrics.get("Kappa", np.nan)
    }

    names = list(selected_metrics.keys())
    values = list(selected_metrics.values())

    plt.figure(figsize=(10, 4))
    plt.bar(names, values)
    plt.ylim(0, 1.05)
    plt.title("Model Performance Metrics")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_decision_boundary_pca_dual(
    X_train,
    y_train,
    alpha,
    b,
    kernel="linear",
    rbf_gamma=1.0
):
    if X_train.shape[1] > 2:
        pca = PCA(n_components=2)
        X_vis = pca.fit_transform(X_train)
    else:
        pca = None
        X_vis = X_train

    x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
    y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid_2d = np.c_[xx.ravel(), yy.ravel()]

    if pca is not None:
        grid_full = pca.inverse_transform(grid_2d)
    else:
        grid_full = grid_2d

    scores = decision_function_dual(
        grid_full,
        X_train,
        alpha,
        y_train,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    z = sigmoid(scores)
    z = z.reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, z, levels=50, alpha=0.8)
    plt.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y_train,
        cmap="bwr",
        edgecolors="black",
        s=45
    )
    plt.title("Decision Boundary in PCA Space")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(label="Predicted Probability")
    plt.show()


def plot_runtime_summary(summary_df):
    if "Runtime_sec" not in summary_df.columns:
        print("Runtime_sec column not found.")
        return

    plt.figure(figsize=(8, 4))
    sns.barplot(data=summary_df, x="Model", y="Runtime_sec")
    plt.title("Runtime Comparison")
    plt.xlabel("Model")
    plt.ylabel("Runtime Second")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def run_all_nl_visualizations(
    state_df,
    K_train,
    X_train,
    y_train,
    X_test,
    y_test,
    y_prob,
    q_values,
    alpha,
    w,
    b,
    metrics,
    selected_feature_names,
    scale,
    kernel,
    rbf_gamma
):
    print("\n========== NL / Solver Visualizations ==========")
    plot_nl_objective_convergence(state_df)
    plot_nl_objective_hist(state_df)
    plot_nl_feasibility(state_df)

    print("\n========== Kernel / Objective Visualizations ==========")
    plot_kernel_matrix_heatmap(K_train, title="Training Kernel Matrix")
    plot_dual_objective_matrix(K_train, y_train, scale)
    plot_dual_interaction_graph(K_train, y_train, scale, max_edges=80)

    print("\n========== Alpha / Support Vector Visualizations ==========")
    plot_q_values(q_values)
    plot_alpha_values(alpha)
    plot_alpha_by_class(alpha, y_train)
    plot_support_vectors_pca(X_train, y_train, alpha)

    print("\n========== Model Performance Visualizations ==========")
    plot_confusion_matrix_custom(y_test, y_prob, threshold=0.5)
    plot_roc_curve_custom(y_test, y_prob)
    plot_pr_curve_custom(y_test, y_prob)
    plot_prediction_distribution(y_test, y_prob)
    plot_metrics_bar(metrics)

    print("\n========== Feature / Decision Boundary Visualizations ==========")

    feature_importance_df = None

    if kernel == "linear" and w is not None:
        feature_importance_df = plot_weight_importance(w, selected_feature_names)
    else:
        print("Feature weight importance is only directly available for linear kernel.")

    plot_decision_boundary_pca_dual(
        X_train,
        y_train,
        alpha,
        b,
        kernel=kernel,
        rbf_gamma=rbf_gamma
    )

    return feature_importance_df


# ============================================================
# MAIN LOOP
# ============================================================

summary = []

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

dual_configs = {
    "DW_NL_QSVM_SoftMargin_LowC": dict(
        kernel=KERNEL_TYPE,
        C=DEFAULT_C,
        n_bits=N_BITS,
        rbf_gamma=RBF_GAMMA
    )
}

total_models = len(dual_configs) * n_splits
completed_jobs = 0

for split in range(1, n_splits + 1):

    print("\n============================================================")
    print(f"Split {split}/{n_splits}")
    print("============================================================")

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * 42
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    print("Train shape:", X_train.shape)
    print("Test shape :", X_test.shape)

    for name, cfg in dual_configs.items():

        start = time.time()

        print("\n------------------------------------------------------------")
        print("Running model:", name)
        print("Config:", cfg)
        print("------------------------------------------------------------")

        model, q, K_train, scale, q_max = build_dual_qsvm_nl(
            X_train,
            y_train,
            n_bits=cfg["n_bits"],
            C=cfg["C"],
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"],
            use_nonzero_alpha_constraint=USE_NONZERO_ALPHA_CONSTRAINT,
            min_total_q=MIN_TOTAL_Q
        )

        print("\nNL model created.")
        print("Number of training variables:", X_train.shape[0])
        print("q_max:", q_max)
        print("alpha scale:", scale)

        try:
            print("Number of model nodes:", model.num_nodes())
        except Exception:
            pass

        try:
            print("Decision state size:", model.decision_state_size())
        except Exception:
            pass

        result, used_time_limit, solver_name = solve_nl_qsvm(
            model,
            token=DWAVE_TOKEN,
            manual_time_limit=MANUAL_NL_TIME_LIMIT,
            label=f"{name}_split_{split}"
        )

        print("\nNL solving completed.")

        try:
            print("Result information:")
            print(result.info)
        except Exception:
            pass

        best_state, best_objective, state_df = select_best_nl_state(model)

        q_values, alpha = extract_alpha_from_nl_state(
            q,
            best_state,
            scale
        )

        y_train_pm = (2 * y_train - 1).astype(float)
        balance_value = np.sum(alpha * y_train_pm)
        true_obj = true_svm_dual_objective(alpha, y_train, K_train)

        print("\nq values:")
        print(q_values)

        print("\nAlpha values:")
        print(alpha)

        print("\nSVM balance value sum(alpha_i * y_i):")
        print(balance_value)

        print("\nTrue SVM dual objective:")
        print(true_obj)

        print("\nNumber of nonzero alpha/support vectors:")
        print(np.sum(alpha > 1e-8))

        b, support_indices = compute_bias_dual(
            alpha,
            y_train,
            K_train,
            C=cfg["C"]
        )

        print("\nBias b:", b)
        print("Support vector indices:", support_indices)

        if cfg["kernel"] == "linear":
            w = extract_linear_weight_from_alpha(alpha, X_train, y_train)
        else:
            w = None

        y_prob = predict_probability_dual(
            X_test,
            X_train,
            alpha,
            y_train,
            b,
            kernel=cfg["kernel"],
            rbf_gamma=cfg["rbf_gamma"]
        )

        metrics = compute_metrics(y_test, y_prob)

        runtime = time.time() - start
        completed_jobs += 1
        progress = (completed_jobs / total_models) * 100

        print(
            f"\n[{progress:6.2f}%] "
            f"{name} | "
            f"Acc={metrics['Accuracy']:.4f} | "
            f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
            f"PR-AUC={metrics['PR-AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Precision={metrics['Precision']:.4f} | "
            f"Recall={metrics['Recall']:.4f} | "
            f"Sensitivity={metrics['Sensitivity']:.4f} | "
            f"Specificity={metrics['Specificity']:.4f} | "
            f"Kappa={metrics['Kappa']:.4f} | "
            f"Runtime={runtime:.2f}s | "
            f"Solver={solver_name} | "
            f"NL_time_limit={used_time_limit}s"
        )

        row = {
            "Model": name,
            "Split": split,
            "Solver_Name": solver_name,
            "Kernel": cfg["kernel"],
            "C": cfg["C"],
            "n_bits": cfg["n_bits"],
            "RBF_Gamma": cfg["rbf_gamma"],
            "Train_Size": X_train.shape[0],
            "Test_Size": X_test.shape[0],
            "Selected_Features": selected_feature_names,
            "q_max": q_max,
            "Alpha_Scale": scale,
            "Accuracy_mean": metrics["Accuracy"],
            "ROC_AUC_mean": metrics["ROC-AUC"],
            "PR_AUC": metrics["PR-AUC"],
            "F1_mean": metrics["F1"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "Kappa": metrics["Kappa"],
            "Runtime_sec": runtime,
            "NL_time_limit_sec": used_time_limit,
            "Best_State": best_state,
            "Best_Objective": best_objective,
            "True_SVM_Objective": true_obj,
            "Balance_Value": balance_value,
            "Balance_Abs": abs(balance_value),
            "Nonzero_Alpha_Count": int(np.sum(alpha > 1e-8))
        }

        summary.append(row)

        pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

        if VISUALIZE:
            feature_importance_df = run_all_nl_visualizations(
                state_df=state_df,
                K_train=K_train,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                y_prob=y_prob,
                q_values=q_values,
                alpha=alpha,
                w=w,
                b=b,
                metrics=metrics,
                selected_feature_names=selected_feature_names,
                scale=scale,
                kernel=cfg["kernel"],
                rbf_gamma=cfg["rbf_gamma"]
            )

            if feature_importance_df is not None:
                print("\nFeature Importance Table:")
                display(feature_importance_df)


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_df = (
    pd.DataFrame(summary)
    .sort_values("Accuracy_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)

if VISUALIZE:
    plot_runtime_summary(summary_df)

print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")